In [ ]:
import os
import sys
import time
import json
from natsort import natsorted
from pathlib import PureWindowsPath, PurePosixPath
import pickle
import numpy as np
import xarray as xr
import pandas as pd
import math 

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.pyplot import figure
from matplotlib.patches import Patch, Rectangle
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import seaborn as sns
import cmasher as cmr

from scipy.signal import find_peaks, peak_widths
import scipy.stats as stats
from scipy.stats import skew, median_abs_deviation
import scikit_posthocs as sp
import statsmodels.api as sm

sys.path.append('../utils') 
from utils_tfc import TFC_proto
from utils_tfc import open_minian, xrconcat_recursive, map_ts
from utils_plot import (
    set_pub_style, get_asterisks, save_metadata_json, 
    lighten_color, add_stat_annotation_two_sided, add_significance_bar)

#General parameters
dpath_cal_all = r'../../data/11.Post_TFC-20' # The directory for TFC-20
dpath_cal_supp = r'../../data/' # The directory for others (TFC-60, TFC-5, TFC-0)
bin_width = 200  # ms
fs = int(1000/bin_width)

colors_anatomy = ['#A6761D', '#845ec2', '#97cebf'] 
colors_beh_i = ['#4091cf', '#e1703c'] # Blue, red
colors_beh_e = ['#4091cf', '#8cba54'] # Blue, green
#plot
dir_output = r'../output_figures'
os.makedirs(dir_output, exist_ok=True)
dir_fig = 'Fig4'
dpath_plot = os.path.join(dir_output, dir_fig)
if not os.path.exists(dpath_plot):
    os.makedirs(dpath_plot)   

## 1.1 Plot of normalized cell proportions with 3 groups in each epoch for conditioning and recall session

In [ ]:
def extract_data_groups_each_trial(df, group_size, group_keys, resp_keys, trial_idx):
    data_group =[]
    for i, resp_k in enumerate(resp_keys):
        data_trials = []
        for j in range(group_size):
            data_trials.append(df.loc[group_keys[j]][resp_k +'_' +str(trial_idx)].values)
        data_group.append(data_trials)
    return data_group

group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_algori_data = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'

ds_all_group_stat =[]
for i in range(group_size):    
    dpath_minian_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_minian_group, test_algori_data)
    print(dpath_test)    
    df_stat = pd.read_csv(os.path.join(dpath_test, "Resp_cells_statistics_cal.csv")) 
    ds_all_group_stat.append(df_stat)

df_stat_all = pd.concat(ds_all_group_stat, keys=group_keys, names=["group", "row"])
resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']
resp_keys_trials = ['cs_first', 'trace_first', 'us_first', 'p1_us_first', 'p2_us_first']
epoch_names=['CS', 'Trace', 'US', 'pUS-1', 'pUS-2']
epoch_dur=[20, 20, 3, 3, 6]

width_mm = 50  # 
height_mm = 30 #    
 # Plot the cell proportions in first 6 trial; two-sides
first_trials_resp_data = extract_data_groups_each_trial(df_stat_all, group_size, group_keys, resp_keys_trials, trial_idx=6)
plot_epoch_proportions_ragged(first_trials_resp_data, width_mm, height_mm, colors_anatomy, group_keys, epoch_names, epoch_dur, dpath_plot,  '01_1_Normalized Resp cell proportions in all 6 trials_conditioning session') 
print('All finished************') 

In [ ]:
# For recall session
def extract_data_groups_each_trial(df, group_size, group_keys, resp_keys, trial_idx):
    data_group =[]
    for i, resp_k in enumerate(resp_keys):
        data_trials = []
        for j in range(group_size):
            data_trials.append(df.loc[group_keys[j]][resp_k +'_' +str(trial_idx)].values)
        data_group.append(data_trials)
    return data_group

group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_algori_data_re = 'post_re_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'
ds_all_group_stat_re =[]
for i in range(group_size):    
    dpath_minian_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_minian_group, test_algori_data_re)
    print(dpath_test)    
    df_stat = pd.read_csv(os.path.join(dpath_test, "Resp_cells_statistics_cal.csv")) 
    ds_all_group_stat_re.append(df_stat)

df_stat_all_re = pd.concat(ds_all_group_stat_re, keys=group_keys, names=["group", "row"])
resp_keys_re = ['tone', 'p_tone']
resp_keys_trials_re = ['tone_first', 'p_tone_first']
epoch_names_re=['CS', 'pCS']
epoch_dur_re=[30, 20]


width_mm = 40  # 
height_mm = 30 #    
 # Plot the cell proportions in first 6 trial; two-sides
first_trials_resp_data_re = extract_data_groups_each_trial(df_stat_all_re, group_size, group_keys, resp_keys_trials_re, trial_idx=4)
plot_epoch_proportions_ragged(first_trials_resp_data_re, width_mm, height_mm, colors_anatomy, group_keys, epoch_names_re, epoch_dur_re, dpath_plot,  'sup_01_1_Normalized Resp cell proportions in all 4 trials_recall session') 
print('All finished************') 

In [ ]:
def plot_epoch_proportions_ragged(groups_data, width_mm, height_mm, colors, labels, epoch_names, durations, output_path, title):
    set_pub_style()         
    # 1. Exact millimeter canvas with constrained layout
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')      
    n_epochs = len(epoch_names)
    base_positions = np.arange(1, n_epochs + 1)   
    
    offsets = [-0.25, 0.0, 0.25]
    box_width = 0.15
    jitter_strength = 0.04
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}        
    # --- 1. PRE-FLIGHT CHECK: Normalize Data & Find Global Max ---
    global_max = 0
    epoch_data_normalized = [] 
    
    for ep_idx in range(n_epochs):
        ep_name = epoch_names[ep_idx]
        duration = durations[ep_idx]
        metadata["Data_Summary"][ep_name] = {}
        
        ep_norm_data = []        
        for grp_idx in range(3):
            group_label = labels[grp_idx]            
            
            # Clean and normalize the data
            d_raw = np.array(groups_data[ep_idx][grp_idx], dtype=float)
            d_clean = d_raw[~np.isnan(d_raw)]
            d_norm = d_clean / duration        
            
            ep_norm_data.append(d_norm)
            
            n_mice = len(d_norm)
            metadata["Data_Summary"][ep_name][group_label] = {
                "N_mice": n_mice,
                "Mean": float(np.mean(d_norm)) if n_mice > 0 else 0,
                "SEM": float(stats.sem(d_norm)) if n_mice > 0 else 0
            }            
            
            if n_mice > 0:
                local_max = np.max(d_norm)
                if local_max > global_max:
                    global_max = local_max
                    
        epoch_data_normalized.append(ep_norm_data)
        
    # 1.3 gives a 30% headroom buffer for double-stacked brackets.
    ax.set_ylim(0, global_max * 1.3) 

    # --- 3. PLOTTING LOOP ---
    for ep_idx in range(n_epochs):
        base_x = base_positions[ep_idx]
        ep_norm_data = epoch_data_normalized[ep_idx]
        
        # A. Plot Box and Scatter
        for grp_idx in range(3):
            d_norm = ep_norm_data[grp_idx]
            if len(d_norm) == 0: continue
            
            x_pos = base_x + offsets[grp_idx]
            color = colors[grp_idx]
            
            # Transparent faces, solid edges
            face_color_rgba = mcolors.to_rgba(color, alpha=0.3)
            
            # Boxplot 
            ax.boxplot(d_norm, positions=[x_pos], widths=box_width, 
                       patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                       medianprops=dict(color=color, linewidth=1.0),
                       whiskerprops=dict(color=color, linewidth=0.5),
                       capprops=dict(color=color, linewidth=0.5))
            
            # Scatter (Tiny points with white borders)
            x_scatter = x_pos + np.random.uniform(-jitter_strength, jitter_strength, size=len(d_norm))
            ax.scatter(x_scatter, d_norm, s=2.5, color=color, 
                       alpha=1.0, edgecolors='white', linewidth=0.25, zorder=3)

        # 2. Add Significance Brackets
        d_ec5b = ep_norm_data[0]
        d_ec3  = ep_norm_data[1]
        d_ca1  = ep_norm_data[2]
        
        local_ep_max = max([np.max(d) if len(d)>0 else 0 for d in ep_norm_data])
        
        x_ec5b = base_x + offsets[0]
        x_ec3  = base_x + offsets[1]
        x_ca1  = base_x + offsets[2]

        
        # Bracket 1: EC5b vs EC3 
        y_next = add_stat_annotation_two_sided(ax, d_ec5b, d_ec3, x_ec5b, x_ec3, local_ep_max, ttest=0, paired=0)        
        if len(d_ec5b) > 0 and len(d_ec3) > 0:
            _, p_val_1 = stats.mannwhitneyu(d_ec5b, d_ec3, alternative='two-sided')
            metadata["Data_Summary"][epoch_names[ep_idx]]["MannWhitney_P_EC5b_vs_EC3"] = float(p_val_1)

        # Bracket 2: EC5b vs CA1
        if y_next is not None:
            add_stat_annotation_two_sided(ax, d_ec5b, d_ca1, x_ec5b, x_ca1, y_next, ttest=0, paired=0)
            
        if len(d_ec5b) > 0 and len(d_ca1) > 0:
            _, p_val_2 = stats.mannwhitneyu(d_ec5b, d_ca1, alternative='two-sided')
            metadata["Data_Summary"][epoch_names[ep_idx]]["MannWhitney_P_EC5b_vs_CA1"] = float(p_val_2)

    # --- FORMATTING ---
    ax.set_xticks(base_positions)
    ax.set_xticklabels(epoch_names, fontsize=7)
    
    # Consolidated to single line to prevent clipping, with labelpad buffer
    ax.set_ylabel('Cell proportion\nNormalized ($s^{-1}$)', labelpad=0.1) 
    
    # REPLACED hardcoded MultipleLocator with dynamic MaxNLocator
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))   
    
    custom_lines = [Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], markersize=4, alpha=1.0) for i in range(3)]
    ax.legend(custom_lines, labels, frameon=False, loc='upper left')    
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Changed from -0.5 to 0.5 to perfectly center the boxes on the canvas
    ax.set_xlim(0.5, n_epochs + 0.5)

    # --- STRICT EXPORTING ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    # Increased w_pad slightly (to 0.05 inches) to give the Y-label physical room to exist
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    #0.05    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 1.2 Plot of EC5b exampel cells (resp cells)

In [ ]:
dpath_cal_group = os.path.join(dpath_cal_all, '01.EC5b')
test_algori_data ='post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

eg_cell_ids = {'CS': {'EC5b-G22-EC1031': [237, 316, 314, 268, 3]}, 
               'Trace': {'EC5b-G22-EC1031':[2, 106, 162, 225, 279]}, 
               'US': {'EC5b-G22-EC1031':[80, 201],
                         'EC5b-G33-EC1620':[132, 181, 162]}, 
               'pUS': {'EC5b-G22-EC1031': [245, 303, 100, 149, 23]}
              }
epochs = {'Base':3 , 'Tone': 20, 'Trace': 20, 'Shock': 3, 'P_Shock': 40}
total_time = sum(epochs.values())
time_vector = np.arange(-3, total_time-3, bin_width/1000)
n_bins = len(time_vector)

groups = list(eg_cell_ids)
base_du = 3 # 10s
post_du = 40 #post shock  use 20s
bins_plot_start = int((20-base_du)*fs) # pre-shock 3s
bins_plot_end = int((20 +20+20+3+ post_du)*fs) # post-shock 17s for plotting

eg_data = []
dpath_cal_bin = os.path.join(dpath_cal_group, test_algori_data)
for i, group in enumerate(groups):
    group_data = []
    eg_epoch = []
    for animal, cell_id in eg_cell_ids[group].items():
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_cal_bin, animal+"_Cal_bin_trial.nc"))                        
        eg_epoch.append(Sig_bin_trial['Sig_each_trial'].sel(unit_id = cell_id, trials=0, bins =range(bins_plot_start, bins_plot_end)).values)
    eg_epoch = np.vstack(eg_epoch)
    eg_data.append(eg_epoch)

width_mm = 60  # ~2.1 inches (Compromise to fit 15 boxes without smudging)
height_mm = 65 # ~1.5 inches    
plot_ec5b_example_cells(eg_data, width_mm, height_mm, groups, time_vector, dpath_plot, '01_2_EC5b_example_responsive_cells')       
print('All finished************') 

In [ ]:
def plot_ec5b_example_cells(eg_data, width_mm, height_mm, groups, time_vector, output_path, title):
    set_pub_style()    
    # 1. Exact millimeter canvas with constrained layout
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained') 
    epoch_names = ['CS', 'Trace', 'US', 'pUS']
    group_colors = ['#4091cf', 'gray', '#e1703c', 'gray'] # Blue, Green, Orange, Purple '#9b59b6'

    vertical_spacing = 16.0 
    current_offset = 0
    n_cells_per_group = 5
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}  
    # ==========================================
    # A. BACKGROUND SHADING FOR EPOCHS
    # ==========================================
    ax.axvspan(0, 20, color=group_colors[0], alpha=0.1, lw=0)          # Tone  
    ax.axvspan(20, 40, color='gray', alpha=0.05, lw=0)           # Trace 
    ax.axvspan(40, 43, color=group_colors[2], alpha=0.15, lw=0)        # Shock 
    ax.axvspan(43, 46, color='gray', alpha=0.05, lw=0)           # Post 
    ax.axvspan(46, 52, color='gray', alpha=0.1, lw=0)           # Post 
    
    y_top = (n_cells_per_group * len(groups)) * vertical_spacing + 2 * vertical_spacing  
    # Fonts reduced to 6 pt for clean fit at 60mm width
    ax.text(10, y_top, epoch_names[0], ha='center', va='bottom', fontsize=7, color='#4091cf', fontweight='bold')
    ax.text(30, y_top,  epoch_names[1], ha='center', va='bottom', fontsize=7, color='gray')
    ax.text(41.5, y_top, epoch_names[2], ha='center', va='bottom', fontsize=7, color='#e1703c', fontweight='bold')
    ax.text(51, y_top,  epoch_names[3], ha='center', va='bottom', fontsize=7, color='gray')
    
    # ==========================================
    # B. PLOT THE TRACES
    # ==========================================
    for group_idx in reversed(range(len(groups))):
        group_name = groups[group_idx]
        color = group_colors[group_idx]
        traces = eg_data[group_idx]
        
        metadata["Data_Summary"][group_name] = {"N_cells_plotted": len(traces)}
        
        group_center_y = current_offset + (len(traces) * vertical_spacing / 2) - (vertical_spacing / 2)
        ax.text(-5, group_center_y, f"{group_name}\nCells", ha='right', va='center', 
                fontsize=7, color=color)
        
        for trace in reversed(traces):
            # Line width reduced to 0.5 to prevent ink bleed at small sizes
            ax.plot(time_vector, trace + current_offset, color='black', lw=0.5)
            current_offset += vertical_spacing            
        current_offset += (vertical_spacing * 0.8)     
    # ==========================================
    # C. SCALE BAR
    # ==========================================
    ax.axis('off')    
    scale_x_len = 10 # 10 seconds
    scale_y_len = 5  # 5 z-scores
    scale_x_pos = 65 # Adjusted to 65s to keep it tighter to the data block
    scale_y_pos = -vertical_spacing 
    
    # Scale bar lines: lw=0.75 for a crisp edge
    ax.plot([scale_x_pos, scale_x_pos + scale_x_len], [scale_y_pos, scale_y_pos], color='black', lw=0.75) 
    ax.plot([scale_x_pos + scale_x_len, scale_x_pos + scale_x_len], [scale_y_pos, scale_y_pos + scale_y_len], color='black', lw=0.75) 
    
    # Scale bar text reduced to 5 pt (Absolute minimum size)
    ax.text(scale_x_pos + scale_x_len/2, scale_y_pos - 1.5, f"{scale_x_len} s", ha='center', va='top', fontsize=5)
    ax.text(scale_x_pos + scale_x_len + 1, scale_y_pos + scale_y_len/2, f"{scale_y_len} z", ha='left', va='center', fontsize=5) 
    # ==========================================
    # D. BOUNDING BOX SAFEGUARDS
    # ==========================================
    # Because axis('off') removes the standard box, constrained_layout might crop the side texts.
    # Explicitly enforce the view limits to encompass all text and scale bars.
    ax.set_xlim(-15, scale_x_pos + scale_x_len + 10)
    ax.set_ylim(scale_y_pos - 5, y_top + 5)
    # ==========================================
    # E. STRICT EXPORTING
    # ==========================================
    base_path = os.path.join(output_path, title.replace(' ', '_'))  
    # Nullify all padding to lock the 60x60mm dimensions
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")   
    plt.savefig(f"{base_path}.png", dpi=300) 
    plt.close()         
    save_metadata_json(metadata, output_path, title)

## 1.3 Lifetime sparness index of EC5b (all cells) in conditioning session

In [ ]:
# In conditioning session
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

flag_raw = 2 # firing sparsity by using deconvolved spikes
test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'
test_algori_data = 'post_01_3_overview_QC_spikes_condi_bin_0.2s_new_z'

spike_data = dict()
for i in range(group_size):   # Use all cells
    dpath_minian_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_minian_group, test_algori_data)
    print(dpath_test)    
    if flag_raw == 2:
        sig_label = 'Spikes'
        Sig_bin_trial_cells = xr.open_dataset(os.path.join(dpath_test, sig_label + "_bin_trials_cells_pool.nc"))    
        Sig_bin_trial_base_cells = xr.open_dataset(os.path.join(dpath_test, sig_label + "_bin_trials_base_cells_pool.nc"))  
        bins_with_base = Sig_bin_trial_base_cells.bins.size + Sig_bin_trial_cells.bins.size
        Sig_bin_trial = xr.concat([Sig_bin_trial_base_cells, Sig_bin_trial_cells], dim='bins').assign_coords({'bins': range(bins_with_base)})#.drop_vars("animal")
        
        Sig_bin_trial = Sig_bin_trial['Sig_each_trial'].values.transpose(1, 0, 2) # To 
        spike_data[group_keys[i]] = Sig_bin_trial

base_du = 20 # 10s
ps1_du = 3 #post-shock1  3s
ps2_du = 6
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+20)*fs)),
    'Trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'US': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pUS-1': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+ps1_du)*fs)), # 
    'pUS-2': (int((base_du+20+20+3+ps1_du)*fs), int((base_du+20+20+3+ps1_du+ps2_du)*fs))}
df_sparsity = analyze_sparsity(spike_data, epochs)

width_mm = 60  # 
height_mm = 30 #  
epoch_names=list(epochs)
colors = {group_keys[0]: colors_anatomy[0], group_keys[1]:  colors_anatomy[1], group_keys[2]:  colors_anatomy[2]} 
plot_sparsity_violin(df_sparsity, 'Sparseness', width_mm, height_mm, epoch_names, group_keys, colors, dpath_plot, '01_3_Lifetime Sparseness of all cells in all trials among epochs')
print('All finished************') 

In [ ]:
def plot_sparsity_violin(df, key_stat, width_mm, height_mm, epoch_order, region_order, colors, output_path, title):
    """
    Generates a high-density violin + scatter plot mathematically locked to exact millimeter dimensions,
    and logs comprehensive cell counts, medians, and Kruskal-Wallis/Dunn stats to a JSON.
    """
    set_pub_style()      
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    # --- 1. PRE-FLIGHT Y-AXIS SCALING ---
    y_min_data = df[key_stat].min()
    y_max_data = df[key_stat].max()        
    # Bottom buffer: 0.05. Top buffer: 20% headroom for double brackets.
    ax.set_ylim(max(0.0, y_min_data - 0.05), y_max_data * 1.2)
    
    # --- 2. DRAW DISTRIBUTIONS (Violin + Strip) ---
    sns.violinplot(
        data=df, x='Epoch', y=key_stat, hue='Region', 
        order=epoch_order, hue_order=region_order, palette=colors,
        cut=0, inner=None, linewidth=0.5, ax=ax, dodge=True, legend=False)        
    for collection in ax.collections: 
        collection.set_alpha(0.4)         
    sns.stripplot(
        data=df, x='Epoch', y=key_stat, hue='Region',
        order=epoch_order, hue_order=region_order, palette=colors,
        dodge=True, alpha=0.3, size=1.0, jitter=0.15, ax=ax, zorder=1, legend=False)
        
    # --- 3. MEDIANS & STATISTICAL TESTING ---
    hue_offsets = {region_order[0]: -0.26, region_order[1]: 0.0, region_order[2]: 0.26}    
    
    for i, epoch in enumerate(epoch_order):
        df_ep = df[df['Epoch'] == epoch]
        if len(df_ep) == 0: continue           
        # Initialize Metadata Structure for this Epoch
        metadata["Statistics"][epoch] = {
            "Group_Data": {},
            "Kruskal_Wallis": None,
            "Dunns_Posthoc": None
        }       
        # A. Calculate and Draw Medians
        for region in region_order:
            subset = df_ep[df_ep['Region'] == region]
            if len(subset) > 0:
                n_cells = len(subset)
                med_val = subset[key_stat].median()                
                # Log Data to JSON
                metadata["Statistics"][epoch]["Group_Data"][region] = {
                    "N_cells": n_cells,
                    "Median": float(med_val)
                }
                
                x_pos = i + hue_offsets[region]                
                ax.scatter(x_pos, med_val, marker='D', color='white', 
                           edgecolor='black', s=4, zorder=5, linewidth=0.5)
                           
        # B. Automated Stats
        vals_grp0 = df_ep[df_ep['Region'] == region_order[0]][key_stat]
        vals_grp1 = df_ep[df_ep['Region'] == region_order[1]][key_stat]
        vals_grp2 = df_ep[df_ep['Region'] == region_order[2]][key_stat]      
        
        # Ensure data in all groups before running Kruskal
        if len(vals_grp0) > 2 and len(vals_grp1) > 2 and len(vals_grp2) > 2:
            stat, p_kw = stats.kruskal(vals_grp0, vals_grp1, vals_grp2)                
            
            # Log Kruskal-Wallis omnibus results
            metadata["Statistics"][epoch]["Kruskal_Wallis"] = {
                "H_statistic": float(stat),
                "p_value": float(p_kw)}            
            if p_kw < 0.05:
                # Dunn's Post-Hoc Test
                p_mat = sp.posthoc_dunn(df_ep, val_col=key_stat, group_col='Region', p_adjust='bonferroni')                                          
                # Compare Group 0 vs Group 1, and Group 0 vs Group 2
                p_0_vs_1 = p_mat.loc[region_order[0], region_order[1]]
                p_0_vs_2 = p_mat.loc[region_order[0], region_order[2]]            
                
                # Log Post-Hoc results to JSON for caption writing
                metadata["Statistics"][epoch]["Dunns_Posthoc"] = {
                    f"{region_order[0]}_vs_{region_order[1]}": float(p_0_vs_1),
                    f"{region_order[0]}_vs_{region_order[2]}": float(p_0_vs_2)}
                
                # Find local roof above the highest point in this specific epoch
                roof = df_ep[key_stat].max()                                
                # Bracket 1 (Inner comparison)
                if p_0_vs_1 < 0.05:
                    roof = add_significance_bar(ax, x1=i + hue_offsets[region_order[1]], x2=i + hue_offsets[region_order[0]], 
                        y_max=roof, text=get_asterisks(p_0_vs_1))                                                
                # Bracket 2 (Outer comparison)
                if p_0_vs_2 < 0.05:
                    add_significance_bar(ax, x1=i + hue_offsets[region_order[2]], x2=i + hue_offsets[region_order[0]], 
                        y_max=roof, text=get_asterisks(p_0_vs_2))

    # --- 4. AESTHETICS & EXPORT ---
    # Formatted Y-labels to prevent text cutoff
    ax.set_ylabel('Lifetime sparseness', labelpad=0.5)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
        
    ax.set_xlabel('')       
    ax.tick_params(axis='x', labelsize=7)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Strict layout padding
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))  
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()       
    # Dump the newly structured metadata dictionary
    save_metadata_json(metadata, output_path, title)

def calculate_treves_rolls_sparseness(spike_vector):
    """
    Calculates Lifetime Sparseness (1 - a) for a 1D array of spikes.
    Values near 1 = Sparse. Values near 0 = Dense.
    """
    if np.sum(spike_vector) == 0:
        return np.nan # Undefined for completely dead cells       
    N = len(spike_vector)
    mean_sq = (np.sum(spike_vector) / N) ** 2
    mean_of_sqs = np.sum(spike_vector ** 2) / N    
    # Add epsilon to prevent division by zero
    a = mean_sq / (mean_of_sqs + 1e-9)
    return 1.0 - a

def analyze_sparsity(spikes_dict, epochs, fs=5.0):
    """
    Extracts sparsity metrics for all brain regions.   
    Parameters:
    -----------
    spikes_dict : dict {'EC5b': array(cells, trials, bins), 'CA1': ...}
    epochs : dict {'Tone': (15, 115), 'Shock': (215, 230)}
    fs : float Sampling rate in Hz (5.0 for 0.2s bins).
    """
    records = []    
    for region, data_3d in spikes_dict.items():
        n_cells, n_trials, n_bins = data_3d.shape       
        for epoch_name, (start, end) in epochs.items():
            # Extract data for this epoch
            # Shape: (n_cells, n_trials, bins_in_epoch)
            epoch_data = data_3d[:, :, start:end]            
            # Flatten trials and bins to get the "Lifetime" vector for each cell
            # Shape: (n_cells, total_epoch_bins)
            lifetime_data = epoch_data.reshape(n_cells, -1)            
            for i in range(n_cells):
                cell_trace = lifetime_data[i, :]              
                # 1. Mean Event Rate (Hz)
                # Sum of spikes / Total time in seconds
                total_time_sec = len(cell_trace) / fs
                mean_rate = np.sum(cell_trace) / total_time_sec                
                # 2. Zero-Inflation (Fraction of silent bins)
                fraction_silent = np.sum(cell_trace == 0) / len(cell_trace)               
                # 3. Treves-Rolls Sparseness
                sparseness = calculate_treves_rolls_sparseness(cell_trace)               
                # Only record cells that fired at least once in the session 
                # (to avoid comparing dead cells)
                if not np.isnan(sparseness):
                    records.append({
                        'Region': region,
                        'Epoch': epoch_name,
                        'MeanRate_Hz': mean_rate,
                        'FractionSilent': fraction_silent,
                        'Sparseness': sparseness})                    
    return pd.DataFrame(records)

## supp_1.3.1 Lifetime sparness index of EC5b (Union Resp cells in paired epochs for all trials) in conditioning and recall session

In [ ]:
# In conditioning session
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_data_dir = 'post_02_2_resp_cal_new_z'
resp_keys = ['tone', 'trace', 'shock', 'p_shock1', 'p_shock2']
ds_groups = [] # using resp cells trial by trial
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_spikes_cell_wise.pkl'), 'rb') as f:
        ds_groups.append(pickle.load(f))        
spike_data_epochs = dict()
for resp_k in resp_keys:
    spike_data = dict()
    for i in range(group_size):
        spike_data[group_keys[i]] = ds_groups[i][resp_k]
    spike_data_epochs[resp_k] = spike_data
#Calculation
base_du = 0 # No based epoch in the resp data
ps1_du = 3 #post-shock1  3s
ps2_du = 10
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+20)*fs)),
    'Trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'US': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pUS-1': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+ps1_du)*fs)), # 
    'pUS-2': (int((base_du+20+20+3+ps1_du)*fs), int((base_du+20+20+3+ps1_du+ps2_du)*fs))}
df_sparsity = analyze_sparsity_per_trial(spike_data_epochs, epochs, resp_keys, fs)


width_mm = 60  # 
height_mm = 30 #  
epoch_names=list(epochs)
#colors = {group_keys[0]: '#e1703c', group_keys[1]: '#4091cf', group_keys[2]: '#8cba54'} # Red-EC5b, Blue-EC3, green-CA1
colors = {group_keys[0]: colors_anatomy[0], group_keys[1]:  colors_anatomy[1], group_keys[2]:  colors_anatomy[2]} 
plot_sparsity_violin(df_sparsity, 'Sparseness', width_mm, height_mm, epoch_names, group_keys, colors, dpath_plot, 'sup_01_3_1_Lifetime Sparseness of resp cells in the paired epochs')
plot_sparsity_violin(df_sparsity, 'FractionSilent', width_mm, height_mm, epoch_names, group_keys, colors, dpath_plot, 'sup_01_3_2_Lifetime FractionSilent of resp cells in the paired epochs')
print('All finished************') 

In [ ]:
# In recall session
test_data_dir = 'post_re_02_2_resp_cal_new_z'
resp_keys = ['tone', 'p_tone']
ds_groups = [] # using resp cells trial by trial
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_spikes_cell_wise.pkl'), 'rb') as f:
        ds_groups.append(pickle.load(f))        
spike_data_epochs = dict()
for resp_k in resp_keys:
    spike_data = dict()
    for i in range(group_size):
        spike_data[group_keys[i]] = ds_groups[i][resp_k]
    spike_data_epochs[resp_k] = spike_data
#Calculation
base_du = 0 # No based epoch in the resp data
post_du = 20 #post-shock1  3s
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+30)*fs)),
    'pCS': (int((base_du+30)*fs),  int((base_du+30+post_du)*fs))}
df_sparsity = analyze_sparsity_per_trial(spike_data_epochs, epochs, resp_keys, fs)


width_mm = 35  # 
height_mm = 30 #  
epoch_names=list(epochs)

colors = {group_keys[0]: colors_anatomy[0], group_keys[1]:  colors_anatomy[1], group_keys[2]:  colors_anatomy[2]} 
plot_sparsity_violin(df_sparsity, 'Sparseness', width_mm, height_mm, epoch_names, group_keys, colors, dpath_plot, 'sup_01_3_3_Lifetime Sparseness of resp cells in the paired epochs_recall session')
print('All finished************') 

In [ ]:
def plot_sparsity_violin(df, key_stat, width_mm, height_mm, epoch_order, region_order, colors, output_path, title):
    """
    Generates a high-density violin + scatter plot mathematically locked to exact millimeter dimensions,
    and logs comprehensive cell counts, medians, and Kruskal-Wallis/Dunn stats to a JSON.
    """
    set_pub_style()      
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    # --- 1. PRE-FLIGHT Y-AXIS SCALING ---
    y_min_data = df[key_stat].min()
    y_max_data = df[key_stat].max()        
    # Bottom buffer: 0.05. Top buffer: 20% headroom for double brackets.
    ax.set_ylim(max(0.0, y_min_data - 0.05), y_max_data * 1.2)
    
    # --- 2. DRAW DISTRIBUTIONS (Violin + Strip) ---
    sns.violinplot(
        data=df, x='Epoch', y=key_stat, hue='Region', 
        order=epoch_order, hue_order=region_order, palette=colors,
        cut=0, inner=None, linewidth=0.5, ax=ax, dodge=True, legend=False)        
    for collection in ax.collections: 
        collection.set_alpha(0.4)         
    sns.stripplot(
        data=df, x='Epoch', y=key_stat, hue='Region',
        order=epoch_order, hue_order=region_order, palette=colors,
        dodge=True, alpha=0.3, size=1.0, jitter=0.15, ax=ax, zorder=1, legend=False)
        
    # --- 3. MEDIANS & STATISTICAL TESTING ---
    hue_offsets = {region_order[0]: -0.26, region_order[1]: 0.0, region_order[2]: 0.26}    
    
    for i, epoch in enumerate(epoch_order):
        df_ep = df[df['Epoch'] == epoch]
        if len(df_ep) == 0: continue           
        # Initialize Metadata Structure for this Epoch
        metadata["Statistics"][epoch] = {
            "Group_Data": {},
            "Kruskal_Wallis": None,
            "Dunns_Posthoc": None
        }       
        # A. Calculate and Draw Medians
        for region in region_order:
            subset = df_ep[df_ep['Region'] == region]
            if len(subset) > 0:
                n_cells = len(subset)
                med_val = subset[key_stat].median()                
                # Log Data to JSON
                metadata["Statistics"][epoch]["Group_Data"][region] = {
                    "N_cells": n_cells,
                    "Median": float(med_val)
                }
                
                x_pos = i + hue_offsets[region]                
                ax.scatter(x_pos, med_val, marker='D', color='white', 
                           edgecolor='black', s=4, zorder=5, linewidth=0.5)
                           
        # B. Automated Stats
        vals_grp0 = df_ep[df_ep['Region'] == region_order[0]][key_stat]
        vals_grp1 = df_ep[df_ep['Region'] == region_order[1]][key_stat]
        vals_grp2 = df_ep[df_ep['Region'] == region_order[2]][key_stat]      
        
        # Ensure data in all groups before running Kruskal
        if len(vals_grp0) > 2 and len(vals_grp1) > 2 and len(vals_grp2) > 2:
            stat, p_kw = stats.kruskal(vals_grp0, vals_grp1, vals_grp2)                
            
            # Log Kruskal-Wallis omnibus results
            metadata["Statistics"][epoch]["Kruskal_Wallis"] = {
                "H_statistic": float(stat),
                "p_value": float(p_kw)}            
            if p_kw < 0.05:
                # Dunn's Post-Hoc Test
                p_mat = sp.posthoc_dunn(df_ep, val_col=key_stat, group_col='Region', p_adjust='bonferroni')                                          
                # Compare Group 0 vs Group 1, and Group 0 vs Group 2
                p_0_vs_1 = p_mat.loc[region_order[0], region_order[1]]
                p_0_vs_2 = p_mat.loc[region_order[0], region_order[2]]            
                
                # Log Post-Hoc results to JSON for caption writing
                metadata["Statistics"][epoch]["Dunns_Posthoc"] = {
                    f"{region_order[0]}_vs_{region_order[1]}": float(p_0_vs_1),
                    f"{region_order[0]}_vs_{region_order[2]}": float(p_0_vs_2)}
                
                # Find local roof above the highest point in this specific epoch
                roof = df_ep[key_stat].max()                                
                # Bracket 1 (Inner comparison)
                if p_0_vs_1 < 0.05:
                    roof = add_significance_bar(ax, x1=i + hue_offsets[region_order[1]], x2=i + hue_offsets[region_order[0]], 
                        y_max=roof, text=get_asterisks(p_0_vs_1))                                                
                # Bracket 2 (Outer comparison)
                if p_0_vs_2 < 0.05:
                    add_significance_bar(ax, x1=i + hue_offsets[region_order[2]], x2=i + hue_offsets[region_order[0]], 
                        y_max=roof, text=get_asterisks(p_0_vs_2))

    # --- 4. AESTHETICS & EXPORT ---
    # Formatted Y-labels to prevent text cutoff
    if key_stat == 'Sparseness':
        ax.set_ylabel('Response sparseness', labelpad=0.5)
    elif key_stat == 'FractionSilent':
        ax.set_ylabel('Fraction of silent bins', labelpad=0.1)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2)) # MaxNLocator(nbins=5)
    #ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
   
    ax.set_xlabel('')       
    ax.tick_params(axis='x', labelsize=7)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Strict layout padding
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))  
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()       
    # Dump the newly structured metadata dictionary
    save_metadata_json(metadata, output_path, title)
    
def calculate_treves_rolls_vectorized(data_matrix):
    """
    Vectorized calculation of Treves-Rolls Sparseness (1 - a) for a 2D matrix.
    data_matrix: shape (n_cells, n_bins)
    """
    # a = (mean(r))^2 / mean(r^2)
    mean_sq = np.mean(data_matrix, axis=1) ** 2
    mean_of_sqs = np.mean(data_matrix ** 2, axis=1)    
    # Add epsilon to prevent division by zero
    a = mean_sq / (mean_of_sqs + 1e-9)
    sparseness = 1.0 - a    
    # Cells that never fired at all should return NaN (undefined sparsity)
    total_spikes = np.sum(data_matrix, axis=1)
    sparseness[total_spikes == 0] = np.nan   
    return sparseness
    
def analyze_sparsity_per_trial(spike_data_epochs, epochs, resp_keys, fs=5.0):
    """
    Extracts sparsity metrics for brain regions with variable cell counts per trial.
    
    Parameters:
    -----------
    spikes_dict : dict
        Format: {'Region': [trial1_matrix, trial2_matrix, ..., trial6_matrix]}
        Where each trialX_matrix is a 2D array of shape (n_cells_in_trial, n_bins).
    epochs : dict
        Format: {'Tone': (start_bin, end_bin), 'Shock': ...}
    fs : float
        Sampling rate in Hz.
    """
    records = []
    for idx, (epoch_name, (start, end)) in enumerate(epochs.items()):
        spikes_dict = spike_data_epochs[resp_keys[idx]]
        for region, trials_list in spikes_dict.items():
            # Iterate through the list of 6 trials
            for trial_idx, trial_data in enumerate(trials_list):         
                # Slice the data for the specific epoch
                # Shape: (n_cells_in_trial, bins_in_epoch)
                epoch_data = trial_data[:, start:end]
                
                if epoch_data.shape[0] == 0:
                    continue # Skip if no cells were recorded in this trial                
                total_time_sec = epoch_data.shape[1] / fs                
                # 1. Mean Event Rate (Hz) -> Vectorized
                mean_rates = np.sum(epoch_data, axis=1) / total_time_sec               
                # 2. Zero-Inflation (Fraction of silent bins) -> Vectorized
                fraction_silent = np.sum(epoch_data == 0, axis=1) / epoch_data.shape[1]                
                # 3. Treves-Rolls Sparseness -> Vectorized
                sparseness_vals = calculate_treves_rolls_vectorized(epoch_data)                
                # Append each valid cell's data to the records
                for i in range(epoch_data.shape[0]):
                    if not np.isnan(sparseness_vals[i]):
                        records.append({
                            'Region': region,
                            'Trial': trial_idx + 1,
                            'Epoch': epoch_name,
                            'MeanRate_Hz': mean_rates[i],
                            'FractionSilent': fraction_silent[i],
                            'Sparseness': sparseness_vals[i]
                        })                        
    return pd.DataFrame(records)

## 1.4 Population sparseness (instantaneous and max) -- all cells(animal-wise)

In [ ]:
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_algori_data = 'post_01_3_overview_QC_spikes_condi_bin_0.2s_new_z'
resp_keys = ['tone', 'trace', 'shock', 'p_shock1', 'p_shock2']

animal_data_dict = dict()
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cal_bin = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_cal_group)
    
    all_items = os.listdir(dpath_cal_bin)
    # Filter for files that contain specific suffix
    target_files = [f for f in all_items if "_2_Spikes_bin_trial.nc" in f]
    sorted_files = natsorted(target_files)
    
    ds_all_animals_cells = []
    for file_name in sorted_files:
        # Construct the full path
        file_path = os.path.join(dpath_cal_bin, file_name)
        Sig_bin_trial = xr.open_dataset(file_path)
        ds_all_animals_cells.append(Sig_bin_trial['Sig_each_trial'].values.transpose(1, 0, 2))
    animal_data_dict[group_keys[i]] = ds_all_animals_cells

# 1. Calculate the DataFrame
base_du = 0 # No based epoch in the resp data
ps1_du = 3 #post-shock1  3s
ps2_du = 10
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+20)*fs)),
    'Trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'US': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pUS-1': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+ps1_du)*fs)), # 
    'pUS-2': (int((base_du+20+20+3+ps1_du)*fs), int((base_du+20+20+3+ps1_du+ps2_du)*fs))}

df_proportions = calculate_population_active_proportion(animal_data_dict, epochs)

width_mm = 50  # 
height_mm = 30 #  
epoch_names=list(epochs)
plot_animal_wise_proportions(df_proportions, 'Active_Proportion', width_mm, height_mm, epoch_names, group_keys, colors_anatomy, dpath_plot, '01_4_Instantiouse poupulation sparseness averaged in all epoch and all trials')
plot_animal_wise_proportions(df_proportions, 'Active_Max_Prop', width_mm, height_mm, epoch_names, group_keys, colors_anatomy, dpath_plot, 'sup_01_4_1_Max poupulation sparseness in all epoch and all trials')

print('All finished************') 

In [ ]:
# For recall
test_algori_data = 'post_re_01_1_overview_QC_spikes_recall_0.2s_new_z'
resp_keys = ['tone', 'p_tone']

animal_data_dict = dict()
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cal_bin = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_cal_group)
    all_items = os.listdir(dpath_cal_bin)
    # Filter for files that contain specific suffix
    target_files = [f for f in all_items if "_2_Spikes_bin_trial.nc" in f]
    sorted_files = natsorted(target_files)
    
    ds_all_animals_cells = []
    for file_name in sorted_files:
        # Construct the full path
        file_path = os.path.join(dpath_cal_bin, file_name)
        Sig_bin_trial = xr.open_dataset(file_path)
        ds_all_animals_cells.append(Sig_bin_trial['Sig_each_trial'].values.transpose(1, 0, 2))
    animal_data_dict[group_keys[i]] = ds_all_animals_cells

# 1. Calculate the DataFrame
base_du = 0 # No based epoch in the resp data
post_du = 20
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+30)*fs)),
    'pCS': (int((base_du+30)*fs),  int((base_du+30+post_du)*fs))}

df_proportions = calculate_population_active_proportion(animal_data_dict, epochs)

width_mm = 30  # 
height_mm = 30 #  
epoch_names=list(epochs)
plot_animal_wise_proportions(df_proportions, 'Active_Proportion', width_mm, height_mm, epoch_names, group_keys, colors_anatomy, dpath_plot, 'sup_01_4_2_Instantiouse poupulation sparseness averaged in all epoch and all trials_recall session')
plot_animal_wise_proportions(df_proportions, 'Active_Max_Prop', width_mm, height_mm, epoch_names, group_keys, colors_anatomy, dpath_plot, 'sup_01_4_3_Max poupulation sparseness in all epoch and all trials_recall session')
print('All finished************') 

In [ ]:
def plot_animal_wise_proportions(df, key_stat, width_mm, height_mm, epoch_order, region_order, colors, output_path, title):
    """
    Plots a pure Matplotlib box + scatter plot for animal-wise active proportions.
    Matches the exact line weights, colored whiskers, and scatter sizing.
    """
    set_pub_style()      
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    
    metadata = {"Figure_Title": title, "Statistics": {}}
    
    # --- 1. SPATIAL SETUP ---
    # Define standard dodging offsets for 3 categories
    offsets = [-0.25, 0.0, 0.25]
    box_width = 0.15
    jitter_strength = 0.04
    base_positions = np.arange(len(epoch_order))
    
    y_min_data = df[key_stat].min()
    y_max_data = df[key_stat].max()    
    ax.set_ylim(max(0.0, y_min_data - 0.02), y_max_data * 1.25)
    
    # --- 2. LOOP THROUGH EPOCHS AND REGIONS ---
    for i, epoch in enumerate(epoch_order):
        df_ep = df[df['Epoch'] == epoch]
        if len(df_ep) == 0: continue
        
        base_x = base_positions[i]       
        metadata["Statistics"][epoch] = {
            "Group_Data": {}, "Kruskal_Wallis": None, "Dunns_Posthoc": None}
        # Arrays to hold data for this epoch's stats
        stat_arrays = []
        # A. Draw Boxplots & Scatters
        for j, region in enumerate(region_order):
            data = df_ep[df_ep['Region'] == region][key_stat].dropna().values
            stat_arrays.append(data)
            
            n_mice = len(data)
            if n_mice > 0:
                metadata["Statistics"][epoch]["Group_Data"][region] = {
                    "N_mice": n_mice,
                    "Median": float(np.median(data)) }
                
                x_pos = base_x + offsets[j]
                color = colors[j]
                # Face is transparent (0.4), edges and whiskers are solid (1.0)
                face_color_rgba = mcolors.to_rgba(color, alpha=0.4)               
                # Pure Matplotlib Boxplot
                ax.boxplot(data, positions=[x_pos], widths=box_width, 
                           patch_artist=True, showfliers=False,
                           boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                           medianprops=dict(color=color, linewidth=1.0),
                           whiskerprops=dict(color=color, linewidth=0.5),
                           capprops=dict(color=color, linewidth=0.5))               
                # Pure Matplotlib Scatter (s=2.5
                x_scatter = x_pos + np.random.uniform(-jitter_strength, jitter_strength, size=n_mice)
                ax.scatter(x_scatter, data, s=2.5, color=color, alpha=1.0, edgecolors='white', linewidth=0.25, zorder=3)

        # B. Automated Statistical Brackets
        vals_grp0, vals_grp1, vals_grp2 = stat_arrays[0], stat_arrays[1], stat_arrays[2]        
        if len(vals_grp0) > 2 and len(vals_grp1) > 2 and len(vals_grp2) > 2:
            stat, p_kw = stats.kruskal(vals_grp0, vals_grp1, vals_grp2)        
            
            metadata["Statistics"][epoch]["Kruskal_Wallis"] = {
                "H_statistic": float(stat), "p_value": float(p_kw)}           
            if p_kw < 0.05:
                p_mat = sp.posthoc_dunn(df_ep, val_col=key_stat, group_col='Region', p_adjust='bonferroni')               
                p_0_vs_1 = p_mat.loc[region_order[0], region_order[1]] 
                p_0_vs_2 = p_mat.loc[region_order[0], region_order[2]]                      
                
                metadata["Statistics"][epoch]["Dunns_Posthoc"] = {
                    f"{region_order[0]}_vs_{region_order[1]}": float(p_0_vs_1),
                    f"{region_order[0]}_vs_{region_order[2]}": float(p_0_vs_2) }
                
                roof = df_ep[key_stat].max()               
                # Bracket 1 (Inner)
                if p_0_vs_1 < 0.05:
                    roof = add_significance_bar(ax, x1=base_x + offsets[0], x2=base_x + offsets[1], 
                        y_max=roof, text=get_asterisks(p_0_vs_1))         
                # Bracket 2 (Outer)
                if p_0_vs_2 < 0.05:
                    add_significance_bar(ax, x1=base_x + offsets[0], x2=base_x + offsets[2], 
                        y_max=roof, text=get_asterisks(p_0_vs_2))

    # --- 3. AXES FORMATTING ---
    ax.set_xticks(base_positions)
    ax.set_xticklabels(epoch_order, fontsize=7)
    # Formatted Y-labels to prevent text cutoff
    if key_stat == 'Active_Proportion':
        ax.set_ylabel('Active cells (%)',  labelpad=0.5)
    elif key_stat == 'Active_Max_Prop':
        ax.set_ylabel('Max active cells (%)', labelpad=0.5)

    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    ax.tick_params(axis='y')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # --- 4. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)        
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    save_metadata_json(metadata, output_path, title)
    
def calculate_population_active_proportion(animal_data_dict, epochs):
    """
    Calculates the Instantaneous Population Active Proportion.
    Math: (Active cells in a single bin) / (Total cells), averaged across all bins in the epoch.    
    Parameters:
    -----------
    animal_data_dict : dict {'Region': [animal1_3d, animal2_3d, ...]} 
        where animalX_3d is (n_cells, n_trials, n_bins)
    epochs : dict {'EpochName': (start_bin, end_bin)}        
    Returns:
    --------
    df : pd.DataFrame
        Animal-wise statistics ready for the box plot.
    """
    records = []   
    for region, animal_list in animal_data_dict.items():
        for animal_idx, animal_data in enumerate(animal_list):            
            # animal_data shape: (n_cells, n_trials, n_total_bins)
            n_cells, n_trials, n_total_bins = animal_data.shape
            
            for epoch_name, (start, end) in epochs.items():
                # 1. Extract the specific time window
                # Shape: (n_cells, n_trials, bins_in_epoch)
                epoch_data = animal_data[:, :, start:end]               
                # 2. Binarize: Is the cell active in this exact bin?
                # Converts spike counts to True/False (1 or 0)
                is_active_per_bin = (epoch_data > 0)               
                # 3. Calculate instantaneous proportion of active cells per bin
                # np.mean across axis=0 (cells) gives the fraction of cells firing 
                # at each exact (trial, bin) coordinate.
                # Shape becomes: (n_trials, bins_in_epoch)
                instantaneous_proportions = np.mean(is_active_per_bin, axis=0)                
                # 4. Average these instantaneous proportions across all bins and trials
                # This yields the average network density for this animal during this epoch
                animal_mean_prop = np.mean(instantaneous_proportions)     
                animal_max_prop = np.max(instantaneous_proportions)  
                records.append({
                    'Region': region,
                    'Animal_ID': f"{region}_M{animal_idx+1}",
                    'Epoch': epoch_name,
                    'Active_Proportion': animal_mean_prop*100,
                    'Active_Max_Prop': animal_max_prop*100
                })               
    return pd.DataFrame(records)

## 1.5 Average activity plot with Zoom-in plot (animal-wise)

In [ ]:
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

base_du = 3 # 10s
post_du = 40 #post shock  use 20s
bins_trial_start = int((20-base_du)*fs) # pre-shock 3s, the whole base is 20s
bins_trial_end = int((20+20+20+3+ post_du)*fs) # post-shock 17s
bins_shock_start = int((20+20+20-base_du)*fs) # pre-shock 3s
bins_shock_end = int((20+20+20+3+ base_du)*fs) # post-shock 17s

cal_trial = dict()
cal_shock = dict()
for i in range(group_size):    
    dpath_minian_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_minian_group, test_algori_data)
    print(dpath_test) 
    Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))  
    Sig_bin_trial = Sig_bin_trial.mean(dim='trials').sel(session='session0_condi')  
    #cal_trial_animal[group_keys[i]] = Sig_bin_trial['Sig_each_trial'].sel(bins=range(bins_trial_start, bins_trial_end)).values

    cal_trial[group_keys[i]] = Sig_bin_trial['Sig_each_trial'].sel(bins=range(bins_trial_start, bins_trial_end)).values
    cal_shock[group_keys[i]] = Sig_bin_trial['Sig_each_trial'].sel(bins=range(bins_shock_start, bins_shock_end)).values  


width_mm = 60  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'pUS-1': 3, 'pUS-2': 6}
plot_trial_population_activity(cal_trial, width_mm, height_mm, epochs, group_keys, colors_anatomy, fs, dpath_plot, '01_5_1_Trial-Averaged Population Activity_animal_wise')
# Plo the zoom in the shock epoch
width_mm = 15  # 
height_mm = 15 #  
epochs_shock =  {'Base':3 , 'US': 3, 'Post':3}
plot_shock_zoom_in(cal_shock, width_mm, height_mm, epochs_shock, group_keys, colors_anatomy, fs, dpath_plot, '01_5_2_Trial-Averaged Population Activity_animal_wise (Shock zoom in)')
print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'pUS-1': ('gray', 0.05),
        'pUS-2': ('gray', 0.1)}       # Post: Light Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)
                        
        metadata["Data_Summary"][grp] = {"N_cells": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel('Mean Z-Score', labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    #ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def plot_shock_zoom_in(cal_shock, width_mm, height_mm, epochs_shock, group_keys, colors, fs, output_path, title):
    """
    Plots a microscopic 20x20mm inset focusing solely on the Shock dynamics.
    Designed to be combined with the main plot in Adobe Illustrator.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')

    # --- 1. TIME VECTOR GENERATION ---
    # For the shock zoom, 0 is typically Shock Onset.
    t_start = -epochs_shock.get('Base', 3.0) 
    
    first_key = [k for k in group_keys if k in cal_shock][0]
    n_bins = cal_shock[first_key].shape[1]
    
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. EPOCH SHADING ---
    # Only shade the US epoch (Orange)
    us_dur = epochs_shock.get('US', 3.0)
    ax.axvspan(0, us_dur, color='#e1703c', alpha=0.15, lw=0, zorder=0)

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_shock: continue
        
        data = cal_shock[grp]
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(data.shape[0])
        color = colors[i]
        
        # Line width dropped to 0.5 because the physical canvas is so small
        ax.plot(x_time, mean_curve, color=color, lw=0.5, zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

    # --- 4. EXTREME MINIMAL FORMATTING ---
    # In a 20mm plot, long labels consume the entire graph. Keep it brief.
    #ax.set_xlabel('Time (s)',  labelpad=0.5)
    #ax.set_ylabel('Z-score', labelpad=0.5)
    
    ax.set_xlim(t_start, x_time[-1])
    
    # Restrict to very few ticks so numbers don't overlap
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    
    ax.tick_params(axis='both', labelsize=5, length=1.5, pad=0.5)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)


    base_path = os.path.join(output_path, title.replace(' ', '_'))  
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()

## supp_1.5.1 Average activity plot (cell-wise) with speed plot 

In [ ]:
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

cal_trial_animal = dict()
speed_trial_animal = []
base_du = 3 # 10s
post_du = 40 # whole ITI 117s-3s (These 3s as the base to next trial)
bins_trial_start = int((20-base_du)*fs) # pre-shock 3s, the whole base is 20s
bins_trial_end = int((20+20+20+3+ post_du)*fs) # post-shock 17s
bins_speed_trial = int((20+160)*fs) # 20s base + 160 trail
for i in range(group_size):    
    dpath_minian_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_minian_group, test_algori_data)
    print(dpath_test)
    Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials_cells_pool.nc"))     
    Sig_bin_trial = Sig_bin_trial['Sig_each_trial'].mean(dim='trials')       
    #Clip data
    cal_trial_animal[group_keys[i]] = Sig_bin_trial.sel(bins=range(bins_trial_start, bins_trial_end)).values

    # Read in speed info
    with open(os.path.join(dpath_test, 'Speed_bin_trials.pkl'), 'rb') as f:
        speed_bin_trial = pickle.load(f)   
    speed_mean_anmials = []
    for key in speed_bin_trial.keys():
        speed_tmp = speed_bin_trial[key].mean(axis=0)
        tmp = np.full(bins_speed_trial, np.nan)
        tmp[:speed_tmp.shape[-1]] = speed_tmp# Average from 6 trials
        speed_mean_anmials.append(tmp[bins_trial_start:bins_trial_end])
    speed_trial_animal.append(np.stack(speed_mean_anmials))
#Merge animal speed data from all groups
speed_trial_animal = np.concatenate(speed_trial_animal, axis=0)

width_mm = 60  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'pUS-1': 3, 'pUS-2': 6}
plot_animal_trial_activity_with_speed(cal_trial_animal, speed_trial_animal, width_mm, height_mm, epochs, group_keys, 
                                      colors_anatomy, fs, dpath_plot, 'sup_01_5_1_Trial-Averaged Population Activity_(cell-wise) with speed ')  
print('All finished************') 

In [ ]:
def plot_animal_trial_activity_with_speed(cal_trial, speed_data, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity (animal-wise) alongside locomotion speed.
    Uses a secondary y-axis for speed, locked to a strict millimeter layout.
    """
    set_pub_style()
    fig, ax1 = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    
    # Create the secondary axis for Speed
    ax2 = ax1.twinx()    
    metadata = {"Figure_Title": title, "Data_Summary": {"Calcium": {}, "Speed": {}}}

    # --- 1. TIME VECTOR GENERATION ---
    t_start = -epochs.get('Base', 3.0) 
    
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    x_time = np.arange(n_bins) / fs + t_start
    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'pUS-1': ('gray', 0.05),
        'pUS-2': ('gray', 0.1)}       # Post: Light Gray      
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue             
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            # Draw on ax1, it will span the whole background
            ax1.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
            
        current_time += ep_dur
    # --- 3. PLOT CALCIUM CURVES (ax1) ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_mice = data.shape[0]  # Animal-wise now
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_mice)
        color = colors[i]        
        
        # Solid colored lines for calcium
        ax1.plot(x_time, mean_curve, color=color, lw=0.75, zorder=3)
        ax1.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                         color=color, alpha=0.3, lw=0, zorder=2)
                         
        metadata["Data_Summary"]["Calcium"][grp] = {
            "N_mice": n_mice, "Peak_Z": float(np.nanmax(mean_curve))   }
    # --- 4. PLOT SPEED CURVE (ax2) ---
    n_mice_speed = speed_data.shape[0]
    speed_mean = np.nanmean(speed_data, axis=0)
    speed_sem = np.nanstd(speed_data, axis=0) / np.sqrt(n_mice_speed)
    
    # Dashed dark gray line for speed to strictly differentiate from calcium
    ax2.plot(x_time, speed_mean, color='#333333', linestyle='--', lw=0.75, zorder=4)
    ax2.fill_between(x_time, speed_mean - speed_sem, speed_mean + speed_sem, 
                     color='#333333', alpha=0.15, lw=0, zorder=3)                     
    metadata["Data_Summary"]["Speed"] = {
        "N_mice": n_mice_speed, "Peak_Speed_cms": float(np.nanmax(speed_mean))  }

    # --- 5. AXES FORMATTING ---
    # Common X-axis
    ax1.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax1.set_xlim(t_start, x_time[-1])
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(20))
    
    # Left Y-axis (Calcium)
    ax1.set_ylabel('Mean Z-Score',labelpad=1)
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    
    # Right Y-axis (Speed)
    ax2.set_ylabel('Speed (cm/s)', labelpad=1, color='#333333')
    ax2.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    
    # Tick params (length=2 to keep ticks microscopic)
    #ax1.tick_params(axis='both', length=2, pad=1)
    ax2.tick_params(axis='y', length=2, pad=1, colors='#333333') # Match tick color to speed line
    
    # Spine Management
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.spines['left'].set_linewidth(0.5)
    ax1.spines['bottom'].set_linewidth(0.5)
    
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_visible(False)
    ax2.spines['right'].set_linewidth(0.5)
    ax2.spines['right'].set_color('#333333') # Match spine color to speed

    # --- 6. EXPORT ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))       
    # Increased w_pad slightly (0.05) to ensure the new right-hand Y-label isn't cropped
    fig.set_constrained_layout_pads(w_pad=0.05, h_pad=0.01, hspace=0, wspace=0)       
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 1.6 Peak latency for responsive cells in conditioning and recall sessions

In [ ]:
#In conditioning session, tone, trace, shock and post shock period of paired resp cells pooled from each trial
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_data_dir = 'post_02_2_resp_cal_new_z'
ds_groups = [] # using resp cells trial by trial
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_cell_wise.pkl'), 'rb') as f:
        ds_groups.append(pickle.load(f))  

base_du = 20 # No based epoch in the resp data
ps1_du = 3 #post-shock1  3s
ps2_du = 6
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+20)*fs)),
    'Trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'US': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'P_US1': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+ps1_du)*fs)), # 
    'P_US2': (int((base_du+20+20+3+ps1_du)*fs), int((base_du+20+20+3+ps1_du+ps2_du)*fs)),
    'P_US': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+ps1_du+ps2_du)*fs))   }


width_mm = 30  # 
height_mm = 30 #  

resp_keys = ['tone', 'trace', 'shock'] #, 'shock'
epochs_dur = {'CS': 20, 'Trace': 20, 'US': 3}
epoch_data = dict()
for idx, resp_key in enumerate(resp_keys):
    bin_s, bin_e = epochs[list(epochs_dur.keys())[idx]]
    for i in range(group_size):
        epoch_data[group_keys[i]] = np.concatenate(ds_groups[i][resp_key], axis=0)[:,bin_s:bin_e]# Pool different trials   
    time_window_sec = list(epochs_dur.values())[idx]
    plot_latency_cdf(epoch_data, time_window_sec, 0, width_mm, height_mm, group_keys, colors_anatomy, fs, dpath_plot, '01_6_'+ str(idx+1)+'_Peak latency of pooled epoch resp cell in the paried epoch-'+resp_key)
# Union of P_US plotting
p_us_data =  dict()
bin_s, bin_e = epochs['P_US']
for i in range(group_size):
    p_us1 = np.concatenate(ds_groups[i]['p_shock1'], axis=0)[:,bin_s:bin_e]
    p_us2 = np.concatenate(ds_groups[i]['p_shock2'], axis=0)[:,bin_s:bin_e]
    p_us_data[group_keys[i]] = np.concatenate([p_us1, p_us2], axis=0)
time_window_sec = ps1_du + ps2_du # shock 3s, 
plot_latency_cdf(p_us_data, time_window_sec, 0, width_mm, height_mm, group_keys, colors_anatomy, fs, dpath_plot, '01_6_4_Peak latency of pooled epoch resp cell in the paried epoch-p_shock')

print('All finished************') 

In [ ]:
#In recall session, tone and post-tone period of paired resp cells pooled from each trial
group_name = ['01.EC5b', '05.EC3-C', '02.CA1-C']
group_keys = ["EC5b", "EC3", "CA1"]
group_size = 3 # 3

test_data_dir = 'post_re_02_2_resp_cal_new_z'
ds_groups = [] # using resp cells trial by trial
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_cell_wise.pkl'), 'rb') as f:
        ds_groups.append(pickle.load(f))  

#Calculation
base_du = 20 # No based epoch in the resp data
post_du = 20 #post-shock1  3s
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+30)*fs)),
    'P_CS': (int((base_du+30)*fs),  int((base_du+30+post_du)*fs))}

width_mm = 40  # 
height_mm = 30 #  

resp_keys = ['tone'] #, 'shock'
epochs_dur = {'CS': 30}
epoch_data = dict()
for idx, resp_key in enumerate(resp_keys):
    bin_s, bin_e = epochs[list(epochs_dur.keys())[idx]]
    for i in range(group_size):
        epoch_data[group_keys[i]] = np.concatenate(ds_groups[i][resp_key], axis=0)[:,bin_s:bin_e]# Pool different trials   
    time_window_sec = list(epochs_dur.values())[idx]
    plot_latency_cdf(epoch_data, time_window_sec, 0, width_mm, height_mm, group_keys, colors_anatomy, fs, dpath_plot, 
                     'sup_01_6_Peak latency of pooled epoch resp cell in the paried epoch-'+resp_key+'_recall')
print('All finished************') 

In [ ]:
def plot_latency_cdf(ds_trial, time_window_sec, flag_para, width_mm, height_mm, region_order, colors, fs, output_path, title):
    """
    Plots a single CDF panel dynamically for 2 or 3 groups.
    - 3 Groups: Plots legend with embedded significance (Bonferroni corrected).
    - 2 Groups: No legend, places a clean significance star in the upper center.
    
    ds_trial: key--group name; value (cell_n, bin_n) cells are pools from all trials
    flag_para=0 -> Plots Peak Latency
    flag_para=1 -> Plots Center of Mass
    """
    set_pub_style()   
    df_metrics = calculate_temporal_metrics(ds_trial, fs, time_window_sec)
    
    fig, ax = plt.subplots(1, 1, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')     
    
    # --- DYNAMIC PARAMETER SELECTION ---
    if flag_para == 0:
        target_metric = 'Peak_Latency'
        x_label = 'Peak latency (s)'
        json_key = 'Peak_Latency_Stats'
    elif flag_para == 1:
        target_metric = 'CoM'
        x_label = 'Center of mass (s)'
        json_key = 'Center_of_Mass_Stats'
    else:
        raise ValueError("flag_para must be 0 (Peak Latency) or 1 (CoM)")

    metadata = {"Figure_Title": title, json_key: {"Cell_Counts": {}, "Statistics": {}}}
    
    # --- 1. EXTRACT DATA & LOG COUNTS DYNAMICALLY ---
    n_groups = len(region_order)
    if n_groups not in [2, 3]:
        raise ValueError("This function expects exactly 2 or 3 regions in region_order.")

    data_list = []
    for r in region_order:
        # Drop NaNs to cleanly extract the array for statistics
        d = df_metrics[df_metrics['Region'] == r][target_metric].dropna()
        data_list.append(d)
        metadata[json_key]["Cell_Counts"][r] = len(d)

    # --- 2. AUTOMATED STATISTICS ---
    legend_labels = list(region_order)
    star_2group = ""
    
    # 3-Group Logic (Omnibus + Post-Hoc)
    if n_groups == 3 and all(len(d) > 2 for d in data_list):
        data_ctrl, data_comp1, data_comp2 = data_list
        stat_kw, p_kw = stats.kruskal(data_ctrl, data_comp1, data_comp2)
        metadata[json_key]["Statistics"]["Kruskal_Wallis"] = {"H": float(stat_kw), "p": float(p_kw)}
        
        if p_kw < 0.05:
            n_comparisons = 2 # Strictly 2 comparisons against Control
            metadata[json_key]["Statistics"]["PostHoc_KS_Bonferroni"] = {}
            
            # Comparison 1 (K-S test)
            _, p_raw1 = stats.ks_2samp(data_ctrl, data_comp1)
            p_adj1 = min(p_raw1 * n_comparisons, 1.0) 
            star1 = get_asterisks(p_adj1)
            metadata[json_key]["Statistics"]["PostHoc_KS_Bonferroni"][f"{region_order[0]}_vs_{region_order[1]}"] = float(p_adj1)
            if star1 and star1 != 'ns':
                legend_labels[1] = f"{region_order[1]} {star1}"
                
            # Comparison 2 (K-S test)
            _, p_raw2 = stats.ks_2samp(data_ctrl, data_comp2)
            p_adj2 = min(p_raw2 * n_comparisons, 1.0)
            star2 = get_asterisks(p_adj2)
            metadata[json_key]["Statistics"]["PostHoc_KS_Bonferroni"][f"{region_order[0]}_vs_{region_order[2]}"] = float(p_adj2)
            if star2 and star2 != 'ns':
                legend_labels[2] = f"{region_order[2]} {star2}"

    # 2-Group Logic (Direct K-S Test)
    elif n_groups == 2 and all(len(d) > 2 for d in data_list):
        data_ctrl, data_comp = data_list
        _, p_val = stats.ks_2samp(data_ctrl, data_comp)
        metadata[json_key]["Statistics"]["KS_Test"] = {f"{region_order[0]}_vs_{region_order[1]}": float(p_val)}
        
        star_2group = get_asterisks(p_val)

    # --- 3. PLOT CDF ---
    sns.ecdfplot(data=df_metrics, x=target_metric, hue='Region', hue_order=region_order, 
                 palette=colors, ax=ax, linewidth=0.75, legend=False)
                 
    # --- 4. FORMATTING ---
    ax.set_xlabel(x_label, labelpad=1)
    ax.set_ylabel('Cumulative fraction', labelpad=0.1)
    
    ax.set_xlim(0, time_window_sec)
    ax.set_ylim(0, 1.05)
    
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. AESTHETICS (LEGEND VS. FLOATING STAR) ---
    if n_groups == 3:
        # Create clean lines for the legend with the updated asterisk strings
        custom_lines = [Line2D([0], [0], color=colors[i], lw=1.5) for i in range(n_groups)]
        ax.legend(custom_lines, legend_labels, frameon=False, loc='lower right', 
                   handlelength=1.5, handletextpad=0.4)   
    elif n_groups == 2:
        # No legend. Place floating significance star in upper center
        if star_2group and star_2group != 'ns':
            # Centers horizontally, floats cleanly at Y=0.95
            ax.text(time_window_sec / 2, 0.75, star_2group, ha='center', va='center', 
                    fontsize=8, color='black')

    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)        
    # Standard opaque backgrounds for clean Illustrator integration
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    # Save precise p-values
    save_metadata_json(metadata, output_path, title)

def calculate_temporal_metrics(data_dict, fs=5.0, time_window_sec=20.0):
    """
    Calculates Peak Latency and Center of Mass (CoM) for each cell.
    Returns a long-format DataFrame ready for Seaborn plotting.
    data_dict: key--group name; value (cell_n, bin_n) cells are pools from all trials
    """
    records = []  
    n_bins = int(time_window_sec * fs)
    time_vector = np.arange(n_bins) / fs 
    
    for region, data in data_dict.items():
        # Ensure only look at the specified time window
        period_data = data[:, :n_bins]              
        # 1. PEAK LATENCY
        peak_bins = np.argmax(period_data, axis=1)
        peak_times = peak_bins / fs        
        
        # 2. CENTER OF MASS (CoM)
        positive_data = np.clip(period_data, 0, None)                
        # Calculate CoM: sum(activity * time) / sum(activity)
        weighted_sum = np.sum(positive_data * time_vector, axis=1)
        total_activity = np.sum(positive_data, axis=1) + 1e-9
        com_times = weighted_sum / total_activity
        
        # Filter out completely silent cells
        active_mask = np.sum(positive_data, axis=1) > 0
        
        for i in range(len(peak_times)):
            if active_mask[i]: 
                records.append({
                    'Region': region,
                    'Peak_Latency': float(peak_times[i]),
                    'CoM': float(com_times[i])})                
    return pd.DataFrame(records)

## 1.7 Heatmap of CS and Trace responsive cells in EC5b and EC3

In [ ]:
group_name = ['01.EC5b', '05.EC3-C'] 
group_keys = ['EC5b', 'EC3' ] 
group_size = len(group_name)

test_algori_cell_info = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'


height_mm = 50 #   
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US': 9}
base_du = 20
base_plot_start = ((base_du - epochs['Base'])* fs)
base_plot_end = ((base_du +40 + 40)* fs)
idx= 7
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_cell_flag = os.path.join(dpath_cal_group, test_algori_cell_info)
    dpath_cal_bin = os.path.join(dpath_cal_group, test_algori_data)

    print(dpath_cal_group)
    # Resp cell info
    with open(os.path.join(dpath_cell_flag, 'resp_cell_flags_cal.pkl'), 'rb') as f:
            dict_group_response = pickle.load(f)    # (n_cells, n_trials)
    
    dict_group_cal = {}
    for key in list(dict_group_response.keys()):
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_cal_bin, f"{key}_Cal_bin_trial.nc"))  
        dict_group_cal[key] = Sig_bin_trial['Sig_each_trial'].values[:, :, base_plot_start:base_plot_end]
    width_mm = 40  # 
    test_key = "cs"
    plot_epoch_subgroup_heatmap(dict_group_response, dict_group_cal, test_key, epochs, fs, width_mm, height_mm, dpath_plot, 
                            f'01_{idx}_1_{test_key} cell activity heatmap-trial 1-{group_keys[i]}')
    width_mm = 45
    test_key = "trace"
    plot_epoch_subgroup_heatmap(dict_group_response, dict_group_cal, test_key, epochs, fs, width_mm, height_mm, dpath_plot, 
                            f'01_{idx}_2_{test_key} cell activity heatmap-trial 1-{group_keys[i]}')
    idx+=1
print('All finished************')      

In [ ]:
def plot_epoch_subgroup_heatmap(dict_group_response, dict_group_cal, resp_key, epochs, fs, width_mm, height_mm, output_path, title):
    """
    Plots a highly optimized heatmap of raw Z-scored calcium traces
    for Trial 1 cells responsive to a specific epoch, pooled across all animals.
    Cells are sorted by their peak activation time within their responsive epoch.
    """
    set_pub_style()    
    # 1. EXTRACT AND POOL DATA FOR TRIAL 1
    t_idx = 0 
    
    pooled_calcium = []
    
    for animal in dict_group_response.keys():
        flags = dict_group_response[animal]
        cal_data = dict_group_cal[animal] # Shape: (trials, cells, bins)
        
        # Get Trial 1 calcium traces for this animal
        cal_t1 = cal_data[t_idx] 
        
        # Get Trial 1 flags for the target epoch (Transpose so shape is (cells,))
        target_flags_t1 = np.array(flags[resp_key]).astype(bool).T[t_idx]
        
        valid_indices = np.where(target_flags_t1)[0]
        
        for idx in valid_indices:
            pooled_calcium.append(cal_t1[idx])

    pooled_calcium = np.array(pooled_calcium)
    if len(pooled_calcium) == 0:
        raise ValueError(f"No '{resp_key}' responsive cells found in Trial 1.")

    # 2. FIND TARGET EPOCH BOUNDARIES FOR SORTING
    current_bin = 0
    target_start = 0
    target_end = 0
    
    # Map the extracted dictionary key to the physical time epoch
    for ep_name, ep_dur in epochs.items():
        ep_bins = int(ep_dur * fs)
        
        match = False
        if resp_key == 'cs' and ep_name == 'CS': match = True
        elif resp_key == 'trace' and ep_name == 'Trace': match = True
        elif resp_key == 'us' and ep_name == 'US': match = True
        #elif resp_key == 'P_US' and ep_name.upper() == 'P_US': match = True
        
        if match:
            target_start = current_bin
            target_end = current_bin + ep_bins
            
        current_bin += ep_bins
        
    # Fallback to CS if parsing somehow misses
    if target_start == target_end:
        target_start = int(epochs.get('Base', 3) * fs)
        target_end = target_start + int(epochs.get('CS', 20) * fs)

    # 3. INTERNAL PEAK SORTING
    target_epoch_data = pooled_calcium[:, target_start:target_end]
    peak_times = np.argmax(target_epoch_data, axis=1)
    
    sort_order = np.argsort(peak_times)
    final_matrix = pooled_calcium[sort_order]

    # 4. PLOTTING THE HEATMAP
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # Calculate time boundaries in seconds to map the X-axis accurately
    base_sec = epochs.get('Base', 3)
    total_sec = final_matrix.shape[1] / fs
    x_start = -base_sec
    x_end = total_sec - base_sec
    n_cells = final_matrix.shape[0]

    # Map the array mathematically to [Time Start, Time End, Bottom Cell, Top Cell]
    im = ax.imshow(final_matrix, aspect='auto', cmap='coolwarm', 
                   vmin=-3, vmax=3, interpolation='nearest', #vmin=-0.5, vmax=3.5
                   extent=[x_start, x_end, n_cells, 0])             
                   
    # 5. AESTHETICS & EPOCH ANNOTATIONS
    # Draw vertical lines for epoch transitions using REAL time (seconds)
    current_time_sec = -base_sec
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time_sec += ep_dur
            continue
        ax.axvline(current_time_sec, color='white', linestyle='--', linewidth=0.5, alpha=0.8)
        current_time_sec += ep_dur
        if ep_name == 'P_US':
            ax.axvline(current_time_sec, color='white', linestyle='--', linewidth=0.5, alpha=0.8)

    # Formatting Axes
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    
    # Format Y-label cleanly (e.g., 'pUS cells' or 'CS cells')
    clean_ylabel = 'pUS' if 'pus' in resp_key.replace('_', '') else resp_key.upper()
    #ax.set_ylabel(f'{clean_ylabel} cells', labelpad=1)
    
    # Set requested intervals: X every 20s, Y every 100 cells
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(20))
    
    ax.tick_params(axis='both',length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # Tiny Colorbar
    if resp_key == 'trace':
        cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.04, anchor=(1.0, 0.05))
        cbar.set_label('Z-Score', labelpad=1)
        cbar.ax.tick_params(labelsize=4.5, length=1, pad=1)
        cbar.outline.set_linewidth(0.5)

    # 6. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf", transparent=True)
    plt.savefig(f"{base_path}.png", dpi=600, transparent=False)
    plt.close()

## 2.1 Basic mean z score plot for EC3-C and EC3-I

In [ ]:
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

base_du = 3 
post_du = 40 
bins_trial_start = int((20-base_du)*fs) 
bins_trial_end = int((20+20+20+3+ post_du)*fs)

cal_trial_animal = dict()
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test) 
    Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))  
    Sig_bin_trial = Sig_bin_trial.mean(dim='trials').sel(session='session0_condi')           
    #Extract data
    cal_trial_animal[group_keys[i]] = Sig_bin_trial['Sig_each_trial'].sel(bins=range(bins_trial_start, bins_trial_end)).values
# 1. plot of Mean Z score
width_mm = 60  # 
height_mm = 30 #  
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8}
plot_trial_population_activity(cal_trial_animal, 'Mean Z-Score',width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, '02_1_Trial-Averaged mean Z score_EC3-animal-wise')
print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    cal_trial : dict()--group name: (n_anmials, n_bins)
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15),
        'P_US3': ('gray', 0.20)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.4))
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    ax.tick_params(axis='both') 
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 2.2 Statistical plot of mean Z score (for US, P-US1, P-US2/3 epochs in EC3)

In [ ]:
def extract_mean_data_among_group(df, group_keys, group_size):
    data_group =[]
    for i in range(group_size):
        data_group.append(df.loc[group_keys[i]].values.mean(axis=1))
    return data_group
    
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_algori_data = 'post_01_8_basic_statistics_new'

test_keys = ['us','p_us1','p_us2' ] #p_us2 includes p_us2 and p_us3 for EC3
key_idx = {k: i+1 for i, k in enumerate(test_keys)}
# initialize containers
ds_cal_z = {k: [] for k in test_keys}
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test)    
    for k in test_keys:
        idx = key_idx[k]        
        # filenames
        cal_file = f"{idx}_Mean z-score trials in epoch-{k}.csv"
        ds_cal_z[k].append(pd.read_csv(os.path.join(dpath_test, cal_file)))
# concat with group keys
for k in test_keys:
    ds_cal_z[k] = pd.concat(ds_cal_z[k], keys=group_keys, names=["group", "row"])

width_mm = 40  # 
height_mm = 30 #  

# 1. Average Z score plotting
x_labels = ['US', 'pUS-1', 'pUS-2/3']
ls_data_us = extract_mean_data_among_group(ds_cal_z['us'], group_keys, group_size)
ls_data_p_us1 = extract_mean_data_among_group(ds_cal_z['p_us1'], group_keys, group_size)
ls_data_p_us2 = extract_mean_data_among_group(ds_cal_z['p_us2'], group_keys, group_size)
plot_multiparameter_z_score([ls_data_us, ls_data_p_us1, ls_data_p_us2], group_keys, x_labels, 'Mean Z-Score', width_mm, height_mm, colors_beh_i, dpath_plot, '02_2_Trial-Averaged mean Z score-statistics with epochs in animal-wise')

print('All finished************') 

In [ ]:
def plot_multiparameter_z_score(data_list, group_labels, x_labels, y_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots a Micro-Panel Facet Grid for multi-epoch comparisons (e.g., US, p-US1, p-US2).
    - Clusters bars tightly together within each epoch panel.
    - Significance brackets dynamically hug the local maximum of each specific epoch.
    - Shares y-limits globally across the row.
    """
    set_pub_style()
    n_epochs = len(data_list)
    n_groups = len(group_labels)
    if n_groups not in [2, 3]:
        raise ValueError("This function is optimized for 2 or 3 animal groups per epoch.")
        
    # --- 1. SPACING LOGIC ---
    # Cluster the boxes closely around the center (0) of the x-axis
    if n_groups == 2:
        positions = [-0.25, 0.25]
        box_width = 0.35
        x_limits = (-0.8, 0.8) # Pads the edges so panels have visual distance between them
    else:
        positions = [-0.35, 0, 0.35]
        box_width = 0.25
        x_limits = (-0.8, 0.8)
    
    # Initialize unified figure layout canvas
    fig, axes = plt.subplots(1, n_epochs, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    if n_epochs == 1: axes = [axes]
    
    metadata = {"Figure_Title": title, "Statistics": {}}

    # --- 2. GLOBAL UNIFIED Y-AXIS SCALING ---
    all_vals_flat = [val for epoch in data_list for grp in epoch for val in grp if not np.isnan(val)]
    if len(all_vals_flat) == 0:
        raise ValueError("No valid scalar data found across parameters.")
        
    global_max = max(all_vals_flat)
    global_min = min(all_vals_flat)
    y_range = global_max - global_min
    
    # Compute shared bounds for the scale itself
    shared_ymin = global_min - (y_range * 0.05)
    shared_ymax = global_max + (y_range * 0.35)

    # --- 3. ITERATE SUBPLOTS THROUGH THE ROW ---
    for idx, ax in enumerate(axes):
        epoch_data = data_list[idx]
        current_x_label = x_labels[idx]
        metadata["Statistics"][current_x_label] = {}
        
        # Apply strict uniform y-constraints to the axis
        ax.set_ylim(shared_ymin, shared_ymax)
        ax.set_xlim(x_limits)
        
        # Calculate LOCAL maximum for this specific epoch to keep brackets close to the data
        local_vals = [val for grp in epoch_data for val in grp if not np.isnan(val)]
        local_max = max(local_vals) if len(local_vals) > 0 else 0
        
        # The gap above the box uses the GLOBAL y_range so the physical gap size looks identical across panels
        current_roof = local_max + (y_range * 0.06)
        
        # A. Plot Background Boxplots and Strip Points
        for grp_idx in range(n_groups):
            grp_data = epoch_data[grp_idx]
            clean_data = grp_data[~np.isnan(grp_data)]
            n_animals = len(clean_data)
            
            metadata["Statistics"][current_x_label][group_labels[grp_idx]] = {"N_animals": n_animals}
            if n_animals == 0: continue
            
            x_pos = positions[grp_idx]
            color = colors[grp_idx]
            face_color_rgba = mcolors.to_rgba(color, alpha=0.4)
            
            ax.boxplot(clean_data, positions=[x_pos], widths=box_width, 
                       patch_artist=True, showfliers=False, zorder=1,
                       boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                       medianprops=dict(color=color, linewidth=1.0),
                       whiskerprops=dict(color=color, linewidth=0.5),
                       capprops=dict(color=color, linewidth=0.5))
            
            jittered_x = np.random.uniform(x_pos - box_width/4, x_pos + box_width/4, size=n_animals)
            ax.scatter(jittered_x, clean_data, s=4, color=color, alpha=1.0, 
                       edgecolors='white', linewidth=0.25, zorder=2)

        # B. Independent Significance Annotations (using the new local positions)
        if len(epoch_data[0]) > 2 and len(epoch_data[1]) > 2:
            current_roof = add_stat_annotation_two_sided(
                ax, epoch_data[0], epoch_data[1], 
                positions[0], positions[1], current_roof, ttest=0, paired=0)            
            _, p_mwu = stats.mannwhitneyu(epoch_data[0], epoch_data[1], alternative='two-sided')
            metadata["Statistics"][current_x_label][f"{group_labels[0]}_vs_{group_labels[1]}"] = float(p_mwu)
            
        if n_groups == 3 and len(epoch_data[0]) > 2 and len(epoch_data[2]) > 2:
            current_roof += (y_range * 0.08)
            add_stat_annotation_two_sided(
                ax, epoch_data[0], epoch_data[2], 
                positions[0], positions[2], current_roof, ttest=0, paired=0)
            
            _, p_mwu = stats.mannwhitneyu(epoch_data[0], epoch_data[2], alternative='two-sided')
            metadata["Statistics"][current_x_label][f"{group_labels[0]}_vs_{group_labels[2]}"] = float(p_mwu)

        # --- 4. MICRO-PANEL FACET GRAPH AESTHETICS ---
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.set_xlabel(current_x_label, labelpad=2)

        #fig.supxlabel('Conditioning epochs')
        # Shared Left Margin Logic
        if idx == 0:
            ax.set_ylabel(y_label, labelpad=1)
            ax.yaxis.set_major_locator(ticker.MultipleLocator(0.4))
            ax.tick_params(axis='y', length=2, pad=1)
            ax.spines['left'].set_linewidth(0.5)
        else:
            ax.set_ylabel('')
            ax.set_yticks([])
            ax.set_yticklabels([])
            ax.tick_params(axis='y', length=0)
            ax.spines['left'].set_visible(False)
            
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT MANUSCRIPT GRAPHIC ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 2.3 Statistical plot of participation Ratio (PR)

In [ ]:
group_name = ['05.EC3-C', '06.EC3-I', '01.EC5b']
group_keys = ['EC3-C', 'EC3-I', 'EC5b']
group_size = len(group_name)

test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

base_du = 3 
post_du = 20 
bins_trial_start = int((20+20+20-base_du)*fs) 
bins_trial_end = int((20+20+20+3+ post_du)*fs) 

cal_trial_cells = []
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test) 
    Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials_cells_pool.nc"))  
    Sig_bin_trial = Sig_bin_trial['Sig_each_trial'].sel(bins=range(bins_trial_start, bins_trial_end))   
    cal_trial_cells.append(Sig_bin_trial.values.transpose(1, 0, 2)) ## shape to (n_cells,n_trials, n_bins)

#Plot the PR curve
ds_c = cal_trial_cells[0]
ds_i = cal_trial_cells[1]
ds_source = cal_trial_cells[2]
t, pr_c, pr_i, rate = analyze_dimensionality_concatenated(ds_c, ds_i, ds_source, fs=5.0, window_sec=3.0, step_sec=0.2) #ec3_i

width_mm = 30  # 
height_mm = 30 #  
epochs = {'Base':3 , 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8} 
plot_pr_constriction(t, pr_c, pr_i, group_keys, epochs, width_mm, height_mm, colors_beh_i, dpath_plot, '02_3_population level-Participation ratio')
print('All finished************')   

In [ ]:
def plot_pr_constriction(time_vec, pr_c, pr_i, group_labels, epochs, width_mm, height_mm, colors, output_path, title):
    """
    Plots the Participation Ratio (Effective Dimensionality) over time.
    Utilizes the pre-calculated time_vec and shifts it so the Base epoch is negative,
    aligning exactly 0 with US onset.
    Locked to strict millimeter layout
    """
    set_pub_style()   
    # 1. Safely flatten inputs to 1D arrays
    time_vec_flat = np.array(time_vec).flatten()
    pr_c_flat = np.array(pr_c).flatten()
    pr_i_flat = np.array(pr_i).flatten()
    
    if not (time_vec_flat.size == pr_c_flat.size == pr_i_flat.size):
        raise ValueError("time_vec, pr_c, and pr_i must have identical sizes.")

    metadata = {
        "Figure_Title": title,
        "Statistics": {
            "N_bins": int(pr_c_flat.size)
        }
    }
    # --- 2. CANVAS SETUP ---
    fig, ax1 = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # --- 3. TIME VECTOR SHIFTING ---
    # Shift the pre-calculated time_vec so the Base epoch becomes negative,
    # placing the US onset perfectly at 0.
    t_base = epochs.get('Base', 3.0)
    shifted_time_vec = time_vec_flat - t_base

    # --- 4. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15),
        'P_US3': ('gray', 0.20)       # Post2: Dark Gray
    }
    
    # Start shading counter at the negative base time (e.g., -3.0)
    current_time = -t_base
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax1.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 5. PLOT DIMENSIONALITY (PR) ---
    ax1.plot(shifted_time_vec, pr_c_flat, color=colors[0], lw=1.0, 
             label=group_labels[0], zorder=3)
    ax1.plot(shifted_time_vec, pr_i_flat, color=colors[1], lw=1.0, 
             label=group_labels[1], zorder=3)

    # --- 6. FORMATTING, TICKS & SPINES ---
    ax1.set_xlabel('Time from US onset', labelpad=1)
    ax1.set_ylabel('Dimensionality (PR)', labelpad=0.1)
    
    # Strict limits based on the actual shifted time vector
    ax1.set_xlim(shifted_time_vec[0], shifted_time_vec[-1])
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
    
    # Limit max ticks to prevent Y-axis crowding in a 30mm height
    ax1.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    
    ax1.tick_params(axis='both', length=2, pad=1, colors='black')
    
    ax1.spines['left'].set_linewidth(0.5)
    ax1.spines['bottom'].set_linewidth(0.5)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)


    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)  
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def get_participation_ratio(data_2d):
    """
    Calculates the Participation Ratio (PR) of a 2D neural population matrix.
    Automatically uses the Gram matrix trick for speed if N_cells > N_timepoints.
    """
    n_cells, n_time = data_2d.shape   
    # 1. Mean-center the data for each cell across the concatenated time vector
    X = data_2d - np.mean(data_2d, axis=1, keepdims=True)  
    # 2. Calculate Covariance Eigenvalues
    if n_cells > n_time:
        # Gram Matrix trick: Eigenvalues of (X.T @ X) are the same as (X @ X.T)
        # Scale doesn't strictly matter for PR since it's a ratio, but keep it mathematically correct.
        cov = np.dot(X.T, X) / (n_time - 1)
    else:
        # Standard Covariance
        cov = np.dot(X, X.T) / (n_time - 1)       
    # Get Eigenvalues (eigh is optimized for symmetric matrices)
    u = np.linalg.eigvalsh(cov)   
    # Filter out numerical noise (tiny or negative float artifacts)
    u = u[u > 1e-9]   
    # 3. PR Formula
    if len(u) == 0:
        return 0.0       
    pr = (np.sum(u) ** 2) / np.sum(u ** 2)
    return pr

def analyze_dimensionality_concatenated(ec3_c, ec3_i, ec5b, fs=5.0, window_sec=3.0, step_sec=0.2):
    """
    Performs sliding-window dimensionality analysis using trial concatenation.
    
    Parameters:
    -----------
    ec3_3d : np.array (n_cells_ec3, n_trials, n_total_bins)
    ec5b_3d : np.array (n_cells_ec5b, n_trials, n_total_bins)
    fs : float
        Sampling rate in Hz (0.2s bins = 5.0 Hz).
    window_sec : float
        Width of the sliding window.
    step_sec : float
        Step size to advance the window.
    """
    n_cells_ec3_c, n_trials, n_total_bins = ec3_c.shape
    n_cells_ec3_i = ec3_i.shape[0]
    window_bins = int(window_sec * fs)
    step_bins = int(step_sec * fs)
    
    time_points = []
    pr_ec3_trace_c = []
    pr_ec3_trace_i = []
    rate_ec5b_trace = []
    
    # Sliding Window Loop
    for t_start in range(0, n_total_bins - window_bins + 1, step_bins):
        t_end = t_start + window_bins
        
        # 1. Slice the 3D window
        ec3_c_window = ec3_c[:, :, t_start:t_end]
        ec3_i_window = ec3_i[:, :, t_start:t_end]
        ec5b_window = ec5b[:, :, t_start:t_end]
        
        # 2. Concatenate Trials (Flatten dimensions 1 and 2)
        # Shape goes from (2000, 6, 15) -> (2000, 90)
        ec3_c_concat = ec3_c_window.reshape(n_cells_ec3_c, -1)
        ec3_i_concat = ec3_i_window.reshape(n_cells_ec3_i, -1)
        # 3. Calculate Dimensionality
        pr = get_participation_ratio(ec3_c_concat)
        pr_ec3_trace_c.append(pr)
        pr = get_participation_ratio(ec3_i_concat)
        pr_ec3_trace_i.append(pr)
        # 4. Calculate EC5b Population Rate
        # Average across all cells, trials, and bins in this window
        rate = np.mean(ec5b_window)
        rate_ec5b_trace.append(rate)
        
        # Record Time (Center of the window)
        time_points.append((t_start + t_end) / 2.0 / fs)
        
    return np.array(time_points), np.array(pr_ec3_trace_c), np.array(pr_ec3_trace_i), np.array(rate_ec5b_trace)

## 2.4 Plot of "braking effect" of EC5b to EC3 (with phase plane trajectory)

In [ ]:
group_name = ['05.EC3-C', '06.EC3-I', '01.EC5b']
group_keys = ['EC3-C', 'EC3-I', 'EC5b']
group_size = len(group_name)

test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

base_du = 3 
post_du = 20 
bins_trial_start = int((20+20+20-base_du)*fs)
bins_trial_end = int((20+20+20+3+ post_du)*fs)

cal_trial_animal = []
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test) 
    Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))
    Sig_bin_trial = Sig_bin_trial.mean(dim='trials').sel(session='session0_condi')           
    cal_trial_animal.append(Sig_bin_trial['Sig_each_trial'].mean(dim='animal').sel(bins=range(bins_trial_start, bins_trial_end)).values)

height_mm = 30 #  
width_mm = 45  # 
colors = ['black','#A6761D'] # 1st is for the delta of I and C, 2nd color is for EC5b
epochs = {'Base':3 , 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8} 
plot_overshoot_brake(cal_trial_animal, group_keys, epochs, fs, width_mm, height_mm, colors, dpath_plot, '02_4_EC5b to EC3 braking effect')
# The Phase-Plane Trajectory
width_mm = 50  # 
epochs = {'Base':3 , 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8} 
plot_phase_plane_trajectory(cal_trial_animal, group_keys, epochs, fs, width_mm, height_mm, colors_beh_i, dpath_plot, 'sup_02_4_EC5b to EC3 phase_plane_trajectory')
print('All finished************') 

In [ ]:
def plot_overshoot_brake(ls_data, group_labels, epochs, fs, width_mm, height_mm, colors, output_path, title):
    """
    Analyzes and plots the "Braking Effect" using the Subtraction Method.
    Calculates Delta Activity (EC3-I - EC3-C) representing "Lost Inhibition" 
    and cross-correlates it directly with the recorded EC5b source activity.
    Locked to strict millimeter layout 
    """
    set_pub_style()
    data_c, data_i, source = ls_data[0], ls_data[1], ls_data[2]
    # 1. Safely flatten inputs to 1D arrays
    target_c = np.array(data_c).flatten()
    target_i = np.array(data_i).flatten()
    source_activity = np.array(source).flatten()
    
    if not (target_c.size == target_i.size == source_activity.size):
        raise ValueError("Time bins for data_c, data_i, and source must be identical sizes.")

    # --- 2. THE SUBTRACTION METHOD CALCULATION ---
    # Delta represents the runaway activity unmasked when the brake is lost
    delta_activity = target_i - target_c
    
    # Compute Pearson cross-correlation coefficient to statistically evaluate coupling
    r_coeff, p_value = stats.pearsonr(delta_activity, source_activity)

    metadata = {
        "Figure_Title": title,
        "Statistics": {
            "Pearson_r": float(r_coeff),
            "Pearson_p": float(p_value),
            "N_bins": int(delta_activity.size)}}
    # --- 3. CANVAS SETUP ---
    fig, ax1 = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    
    # Establish time vector. Shifted so 0 aligns with Shock Onset
    time_vec = np.arange(delta_activity.size) / fs - 3.0

    # --- 4. DUAL AXIS PLOTTING ---
    # Use twinx to overlay both variables without squashing their distinct amplitudes
    ax2 = ax1.twinx()    
    # Plot Delta Activity on the Left Axis (ax1)
    ax1.plot(time_vec, delta_activity, color=colors[0], lw=0.75, 
             label=f"$\Delta$ ({group_labels[1]} - {group_labels[0]})", zorder=3)    
    # Plot EC5b Source Activity on the Right Axis (ax2)
    ax2.plot(time_vec, source_activity, color=colors[1], lw=0.75, 
             label=f"{group_labels[2]} Activity", zorder=3)

    # --- 5. OVERLAYS & REFERENCE LINES ---
    t_start = -epochs.get('Base', 3.0)     
    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
            'US': ('#e1703c', 0.15),      # Shock: Orange
            'P_US1': ('gray', 0.05),      # Post1: Light Gray
            'P_US2': ('gray', 0.15),
            'P_US3': ('gray', 0.20)}       # Postw: dark Gray       
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax1.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur
        
    # Zero baseline reference for the Delta Activity
    ax1.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.8, zorder=1)

    # Print the r-value overlay securely inside the plot canvas window
    p_string = f"p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"
    stat_text = f"Pearson $r$ = {r_coeff:.2f}\n{p_string}"
    ax1.text(0.35, 0.90, stat_text, transform=ax1.transAxes, 
             fontsize=6,  va='top', ha='left', zorder=4)

    # --- 6. FORMATTING, TICKS & SPINES ---
    ax1.set_xlabel('Time from US onset (s)',  labelpad=1)
    ax1.set_ylabel('$\Delta$ Activity (Z-Score)', color=colors[0], labelpad=0.1)
    ax2.set_ylabel(f'{group_labels[2]} (Z-Score)', color=colors[1], labelpad=2)
    
    ax1.set_xlim(time_vec[0], time_vec[-1])
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
    
    # Limit max ticks to prevent Y-axis crowding
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax2.yaxis.set_major_locator(ticker.MultipleLocator(0.1)) # axNLocator(nbins=4)
    
    # Match spine and tick colors to their respective plot line identities
    ax1.tick_params(axis='y', length=2, pad=1, colors=colors[0])
    ax1.tick_params(axis='x', length=2, pad=1, colors='black') # X-axis remains black
    ax2.tick_params(axis='y', length=2, pad=1, colors=colors[1])
    
    ax1.spines['left'].set_color(colors[0])
    ax1.spines['left'].set_linewidth(0.5)
    ax2.spines['right'].set_color(colors[1])
    ax2.spines['right'].set_linewidth(0.5)
    ax1.spines['bottom'].set_linewidth(0.5)
    
    # Clean up upper and inner boundaries
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_visible(False) # Prevent bottom spine double-drawing

    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    # Saved with default opaque backdrops as requested
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()    
    save_metadata_json(metadata, output_path, title)

def plot_phase_plane_trajectory(ls_data, group_labels, epochs, fs, width_mm, height_mm, colors, output_path, title):
    """
    Plots the Phase-Plane Trajectory (State Space) demonstrating dynamic network coupling.
    X-axis: EC5b (Source) Activity
    Y-axis: EC3 (Target) Activity
    Shows how the wild-type circuit is "pulled" by inhibition, and how the manipulated 
    circuit escapes this trajectory.
    Locked to strict millimeter layout
    """
    set_pub_style()
    data_c, data_i, source = ls_data[0], ls_data[1], ls_data[2]
    
    # 1. Safely flatten inputs to 1D arrays
    target_c = np.array(data_c).flatten()
    target_i = np.array(data_i).flatten()
    source_activity = np.array(source).flatten()
    
    if not (target_c.size == target_i.size == source_activity.size):
        raise ValueError("Time bins for data_c, data_i, and source must be identical sizes.")

    # --- 2. DYNAMIC TIME SLICING (Extracting US to P_US2) ---
    # Reconstruct the time vector to find the exact bins for the 9-second window
    t_start = -epochs.get('Base', 3.0) 
    time_vec = np.arange(target_c.size) / fs + t_start
    
    window_start_time = None
    window_end_time = None
    current_time = t_start
    
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'US':
            window_start_time = current_time
        if ep_name == 'P_US2':
            window_end_time = current_time + ep_dur
        current_time += ep_dur
        
    # Fallback to standard 0 to 9s if epoch names differ
    if window_start_time is None: window_start_time = 0.0
    if window_end_time is None: window_end_time = 9.0

    # Create boolean mask for the target window
    mask = (time_vec >= window_start_time) & (time_vec <= window_end_time)
    
    x_source = source_activity[mask]
    y_target_c = target_c[mask]
    y_target_i = target_i[mask]

    metadata = {
        "Figure_Title": title,
        "Statistics": {
            "Trajectory_Window_Start": float(window_start_time),
            "Trajectory_Window_End": float(window_end_time),
            "N_bins_plotted": int(x_source.size)
        }
    }

    # --- 3. CANVAS SETUP ---
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    

    # --- 4. PHASE-PLANE PLOTTING ---
    # Plot EC3-C (Wild-Type) Trajectory
    ax.plot(x_source, y_target_c, color=colors[0], lw=1.0, alpha=0.9, 
             label=f"{group_labels[0]} Trajectory", zorder=3)
    
    # Plot EC3-I (Manipulated) Trajectory to show the "escape"
    ax.plot(x_source, y_target_i, color=colors[1], lw=1.0, linestyle='--', alpha=0.9, 
             label=f"{group_labels[1]} Trajectory", zorder=2)

    # --- 5. ADDING TIME-DIRECTION ARROWS ---
    # Add arrows evenly spaced along the line to show how the system evolves over time
    num_arrows = 3
    step = len(x_source) // (num_arrows + 1)
    
    for i in range(step, len(x_source) - step, step):
        # Arrow for Control Group
        ax.annotate('', xy=(x_source[i+1], y_target_c[i+1]), 
                    xytext=(x_source[i], y_target_c[i]),
                    arrowprops=dict(arrowstyle="->", color=colors[0], lw=1.5, shrinkA=0, shrinkB=0), zorder=4)
        
        # Arrow for Inhibition Group
        ax.annotate('', xy=(x_source[i+1], y_target_i[i+1]), 
                    xytext=(x_source[i], y_target_i[i]),
                    arrowprops=dict(arrowstyle="->", color=colors[1], lw=1.5, shrinkA=0, shrinkB=0), zorder=4)
                    
    # Mark the precise start point (US onset) of the trajectory with a clear dot
    ax.scatter(x_source[0], y_target_c[0], color=colors[0], s=8, zorder=5)
    ax.scatter(x_source[0], y_target_i[0], color=colors[1], s=8, zorder=5)
    ax.text(x_source[0],  0, 'Start (US)', fontsize=7,  ha='center', va='bottom', zorder=5) #x_source[0], max(y_target_c[0], y_target_i[0]) + 0.2

    # --- 6. FORMATTING, TICKS & SPINES ---
    # X and Y axes are directly cross-referenced Brain Regions
    ax.set_xlabel(f'{group_labels[2]} (Z-Score)',  labelpad=1)
    ax.set_ylabel('EC3 (Z-Score)',  labelpad=1)
    
    # Reference baselines defining standard activity
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
    ax.axvline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
    
    # Tick management (Max 4-5 ticks keeps it concise)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(0.05))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax.tick_params(axis='both', length=2, pad=1)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 7. EXPORT GRAPHICS ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    
    # Saved with default opaque backdrops as requested
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()    
    
    save_metadata_json(metadata, output_path, title)

## 2.5 Plot of mean z score for epoch resp cells (CS, trace) in all trials

In [ ]:
def extract_PETH_mean_all_trials(cal_data, resp_name, bin_s, bin_e, trial_num):
    #cal_data-- e.g.: ds_animal_wise['tone'] =  [animal_1, animal_2, ... animal_n ] for each animal
    #  For each animal : [np.array(n_cells_1, n_bins),np.array(n_cells_2, n_bins)...np.array n_cells_6, n_bins) ] 
    # data_group--(n_animals, n_bins)
    animal_num = len(cal_data[resp_name])
    data_group =[]
    for i in range(animal_num):
        data_trials = []
        for trial_idx in range(trial_num):        
            tmp = cal_data[resp_name][i][trial_idx]
            if not np.isnan(tmp).all():
                data_trials.append(tmp) #[:, bin_s:bin_e].mean(axis=0))        
        if len(data_trials) > 0:
            data_trials = np.concatenate(data_trials, axis=0)
            data_trials = np.nanmean(data_trials, axis=0)[bin_s:bin_e]
            data_group.append(data_trials)
    return np.array(data_group)
    
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_data_dir = 'post_02_2_resp_cal_new_z'

base_du = 3 
post_du = 40 
bin_s = int((20-base_du)*fs) 
bin_e = int((20+20+20+3+ post_du)*fs)
trial_num = 6

width_mm = 50  
height_mm = 30 #
epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6,  'P_US3': 8}

# 1. PETH of CS resp cells (Averaged on all 6 trials)
resp_name = 'tone' 
cal_trial_animal = dict()
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)
    cal_trial_animal[group_keys[i]] = extract_PETH_mean_all_trials(cal_tmp, resp_name, bin_s, bin_e, trial_num)
plot_trial_population_activity(cal_trial_animal, 'Mean Z-Score',width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, '02_5_1_Trial-Averaged mean Z score-(CS resp cells)-animal-wise')

# 1. PETH of trace resp cells (Averaged on all 6 trials)
resp_name = 'trace' 
cal_trial_animal = dict()
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)
    cal_trial_animal[group_keys[i]] = extract_PETH_mean_all_trials(cal_tmp, resp_name, bin_s, bin_e, trial_num)
plot_trial_population_activity(cal_trial_animal, 'Mean Z-Score',width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, '02_5_2_Trial-Averaged mean Z score-(Trace resp cells)-animal-wise')

# 2. PETH of US resp cells (Averaged on all 6 trials)
resp_name = 'shock'
cal_trial_animal = dict()
for i in range(group_size):    
    dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
    with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_animal_wise.pkl'), 'rb') as f:
        cal_tmp = pickle.load(f)
    cal_trial_animal[group_keys[i]] = extract_PETH_mean_all_trials(cal_tmp, resp_name, bin_s, bin_e, trial_num)
plot_trial_population_activity(cal_trial_animal, 'Mean Z-Score',width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, 'sup_02_5_3_Trial-Averaged mean Z score-(Shock resp cells)-animal-wise')

print('All finished************') 

In [ ]:
def plot_trial_population_activity(cal_trial, y_label, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout.
    cal_trial : dict()--group name: (n_anmials, n_bins)
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15),
        'P_US3': ('gray', 0.20)}       # Postw: dark Gray        
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel(y_label, labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    if y_label == 'Mean Z-Score':
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5)) # MultipleLocator(0.4)
    else:
        ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    #ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=6))
    ax.tick_params(axis='both') #, length=2, pad=1
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## 2.6 Peak latency in CS, Trace, US and pUS period of paired resp cells pooled from all trials

In [ ]:
# For visulization of CDF of resp cells
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_data_dir = 'post_02_2_resp_cal_new_z'
height_mm = 30  

base_du = 20 # No based epoch in the resp data
ps1_du = 3 # post-shock1  3s
ps2_du = 6+8 # both ps2 and ps3
epochs = {
    'CS':  (int(base_du*fs),  int((base_du+20)*fs)),
    'Trace': (int((base_du+20)*fs),  int((base_du+20+20)*fs)),
    'US': (int((base_du+20+20)*fs), int((base_du+20+20+3)*fs)),
    'pUS-1': (int((base_du+20+20+3)*fs), int((base_du+20+20+3+ps1_du)*fs)), # 
    'pUS-2': (int((base_du+20+20+3+ps1_du)*fs), int((base_du+20+20+3+ps1_du+ps2_du)*fs))}

resp_keys = ['tone', 'trace', 'shock', 'p_shock1', 'p_shock2_both'] #, , 'trace', 'shock'
epochs_dur = {'CS': 20, 'Trace': 20, 'US': 3, 'pUS-1': 3, 'pUS-2': 14}

epoch_data = dict()
for idx, resp_key in enumerate(resp_keys):
    bin_s, bin_e = epochs[list(epochs_dur.keys())[idx]]
    for i in range(group_size):
        dpath_data_group = os.path.join(dpath_cal_all, test_data_dir)
        with open(os.path.join(dpath_data_group, group_name[i] + '_ds_cal_cell_wise.pkl'), 'rb') as f:
            cal_tmp = pickle.load(f)
        
        epoch_data[group_keys[i]] = np.concatenate(cal_tmp[resp_key], axis=0)[:, bin_s:bin_e]# Pool different trials   
    time_window_sec = list(epochs_dur.values())[idx]
    width_mm = 30  # 
    if resp_key == 'p_shock2_both':
        width_mm = 40
    plot_latency_cdf(epoch_data, time_window_sec, 0, width_mm, height_mm, group_keys, colors_beh_i, fs, dpath_plot, f'02_6_{str(idx+1)}_Peak latency of pooled epoch resp cell in the paried epoch_{resp_key}')
print('All finished************') 

In [ ]:
def plot_latency_cdf(ds_trial, time_window_sec, flag_para, width_mm, height_mm, region_order, colors, fs, output_path, title):
    """
    Plots a single CDF panel dynamically for 2 or 3 groups.
    - 3 Groups: Plots legend with embedded significance (Bonferroni corrected).
    - 2 Groups: No legend, places a clean significance star in the upper center.
    
    ds_trial: key--group name; value (cell_n, bin_n) cells are pools from all trials
    flag_para=0 -> Plots Peak Latency
    flag_para=1 -> Plots Center of Mass
    """
    set_pub_style()   
    df_metrics = calculate_temporal_metrics(ds_trial, fs, time_window_sec)
    
    fig, ax = plt.subplots(1, 1, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')     
    
    # --- DYNAMIC PARAMETER SELECTION ---
    if flag_para == 0:
        target_metric = 'Peak_Latency'
        x_label = 'Peak latency (s)'
        json_key = 'Peak_Latency_Stats'
    elif flag_para == 1:
        target_metric = 'CoM'
        x_label = 'Center of mass (s)'
        json_key = 'Center_of_Mass_Stats'
    else:
        raise ValueError("flag_para must be 0 (Peak Latency) or 1 (CoM)")

    metadata = {"Figure_Title": title, json_key: {"Cell_Counts": {}, "Statistics": {}}}
    
    # --- 1. EXTRACT DATA & LOG COUNTS DYNAMICALLY ---
    n_groups = len(region_order)
    if n_groups not in [2, 3]:
        raise ValueError("This function expects exactly 2 or 3 regions in region_order.")

    data_list = []
    for r in region_order:
        # Drop NaNs to cleanly extract the array for statistics
        d = df_metrics[df_metrics['Region'] == r][target_metric].dropna()
        data_list.append(d)
        metadata[json_key]["Cell_Counts"][r] = len(d)

    # --- 2. AUTOMATED STATISTICS ---
    legend_labels = list(region_order)
    star_2group = ""
    
    # 3-Group Logic (Omnibus + Post-Hoc)
    if n_groups == 3 and all(len(d) > 2 for d in data_list):
        data_ctrl, data_comp1, data_comp2 = data_list
        stat_kw, p_kw = stats.kruskal(data_ctrl, data_comp1, data_comp2)
        metadata[json_key]["Statistics"]["Kruskal_Wallis"] = {"H": float(stat_kw), "p": float(p_kw)}
        
        if p_kw < 0.05:
            n_comparisons = 2 # Strictly 2 comparisons against Control
            metadata[json_key]["Statistics"]["PostHoc_KS_Bonferroni"] = {}
            
            # Comparison 1 (K-S test)
            _, p_raw1 = stats.ks_2samp(data_ctrl, data_comp1)
            p_adj1 = min(p_raw1 * n_comparisons, 1.0) 
            star1 = get_asterisks(p_adj1)
            metadata[json_key]["Statistics"]["PostHoc_KS_Bonferroni"][f"{region_order[0]}_vs_{region_order[1]}"] = float(p_adj1)
            if star1 and star1 != 'ns':
                legend_labels[1] = f"{region_order[1]} {star1}"
                
            # Comparison 2 (K-S test)
            _, p_raw2 = stats.ks_2samp(data_ctrl, data_comp2)
            p_adj2 = min(p_raw2 * n_comparisons, 1.0)
            star2 = get_asterisks(p_adj2)
            metadata[json_key]["Statistics"]["PostHoc_KS_Bonferroni"][f"{region_order[0]}_vs_{region_order[2]}"] = float(p_adj2)
            if star2 and star2 != 'ns':
                legend_labels[2] = f"{region_order[2]} {star2}"

    # 2-Group Logic (Direct K-S Test)
    elif n_groups == 2 and all(len(d) > 2 for d in data_list):
        data_ctrl, data_comp = data_list
        _, p_val = stats.ks_2samp(data_ctrl, data_comp)
        metadata[json_key]["Statistics"]["KS_Test"] = {f"{region_order[0]}_vs_{region_order[1]}": float(p_val)}
        
        star_2group = get_asterisks(p_val)

    # --- 3. PLOT CDF ---
    sns.ecdfplot(data=df_metrics, x=target_metric, hue='Region', hue_order=region_order, 
                 palette=colors, ax=ax, linewidth=0.75, legend=False)
                 
    # --- 4. FORMATTING ---
    ax.set_xlabel(x_label, labelpad=1)
    ax.set_ylabel('Cumulative fraction', labelpad=0.1)
    
    ax.set_xlim(0, time_window_sec)
    ax.set_ylim(0, 1.05)
    
    ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. AESTHETICS (LEGEND VS. FLOATING STAR) ---
    if n_groups == 3:
        # Create clean lines for the legend with the updated asterisk strings
        custom_lines = [Line2D([0], [0], color=colors[i], lw=1.5) for i in range(n_groups)]
        ax.legend(custom_lines, legend_labels, frameon=False, loc='lower right', 
                  handlelength=1.5, handletextpad=0.4)   
    elif n_groups == 2:
        # No legend. Place floating significance star in upper center
        if star_2group and star_2group != 'ns':
            # Centers horizontally, floats cleanly at Y=0.95
            ax.text(time_window_sec / 2, 0.85, star_2group, ha='center', va='center', 
                    fontsize=8, color='black')

    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)        
    # Standard opaque backgrounds for clean Illustrator integration
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def calculate_temporal_metrics(data_dict, fs=5.0, time_window_sec=20.0):
    """
    Calculates Peak Latency and Center of Mass (CoM) for each cell.
    Returns a long-format DataFrame ready for Seaborn plotting.
    data_dict: key--group name; value (cell_n, bin_n) cells are pools from all trials
    """
    records = []  
    n_bins = int(time_window_sec * fs)
    time_vector = np.arange(n_bins) / fs 
    
    for region, data in data_dict.items():
        # Ensure only look at the specified time window
        period_data = data[:, :n_bins]              
        # 1. PEAK LATENCY
        peak_bins = np.argmax(period_data, axis=1)
        peak_times = peak_bins / fs        
        
        # 2. CENTER OF MASS (CoM)
        positive_data = np.clip(period_data, 0, None)                
        # Calculate CoM: sum(activity * time) / sum(activity)
        weighted_sum = np.sum(positive_data * time_vector, axis=1)
        total_activity = np.sum(positive_data, axis=1) + 1e-9
        com_times = weighted_sum / total_activity
        
        # Filter out completely silent cells
        active_mask = np.sum(positive_data, axis=1) > 0
        
        for i in range(len(peak_times)):
            if active_mask[i]: 
                records.append({
                    'Region': region,
                    'Peak_Latency': float(peak_times[i]),
                    'CoM': float(com_times[i])})                
    return pd.DataFrame(records)

## supp_2.7 Proportion of Resp cell for each epoch from all trials

In [ ]:
def extract_data_groups_each_trial(df, group_size, group_keys, resp_keys, trial_idx):
    data_group =[]
    for i, resp_k in enumerate(resp_keys):
        data_trials = []
        for j in range(group_size):
            data_trials.append(df.loc[group_keys[j]][resp_k +'_' +str(trial_idx)].values *100) # to %
        data_group.append(data_trials)
    return data_group

group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_algori_data = 'post_02_1_resp_MI_reg_bootstrap_bin_200ms_strict_safe_zone_epoch_base_sig'

ds_all_group_stat =[]
for i in range(group_size):    
    dpath_minian_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_minian_group, test_algori_data)
    print(dpath_test)    
    df_stat = pd.read_csv(os.path.join(dpath_test, "Resp_cells_statistics_cal.csv")) 
    ds_all_group_stat.append(df_stat)

df_stat_all = pd.concat(ds_all_group_stat, keys=group_keys, names=["group", "row"])
resp_keys = ['cs', 'trace', 'us', 'p1_us', 'p2_us']
resp_keys_trials = ['cs_first', 'trace_first', 'us_first', 'p1_us_first', 'p2_us_first']
epoch_names=['CS', 'Trace', 'US', 'pUS-1', 'pUS-2']

width_mm = 40  # 
height_mm = 30 #    
# Plot the cell proportions in first 6 trial; two-sides
first_trials_resp_data = extract_data_groups_each_trial(df_stat_all, group_size, group_keys, resp_keys_trials, trial_idx=6)
plot_epoch_proportion_no_norm(first_trials_resp_data, width_mm, height_mm, colors_beh_i, group_keys, epoch_names, dpath_plot,  'sup_02_7_Epoch Resp cell proportions in all 6 trials') 
print('All finished************') 

In [ ]:
def plot_epoch_proportion_no_norm(groups_data, width_mm, height_mm, colors, labels, epoch_names, output_path, title):
    set_pub_style()         
    # 1. Exact millimeter canvas with constrained layout
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')      
    n_epochs = len(epoch_names)
    n_groups = len(labels)
    base_positions = np.arange(1, n_epochs + 1)   
    
    # MODIFIED: Offsets for exactly 2 groups (centers them around the tick)
    offsets = [-0.15, 0.15]
    box_width = 0.15
    jitter_strength = 0.04
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}        
    # --- 1. PRE-FLIGHT CHECK: Normalize Data & Find Global Max ---
    global_max = 0
    epoch_data_normalized = [] 
    
    for ep_idx in range(n_epochs):
        ep_name = epoch_names[ep_idx]
        metadata["Data_Summary"][ep_name] = {}
        
        ep_norm_data = []        
        for grp_idx in range(n_groups):
            group_label = labels[grp_idx]            
            
            # Clean and normalize the data
            d_raw = np.array(groups_data[ep_idx][grp_idx], dtype=float)
            d_clean = d_raw[~np.isnan(d_raw)]
            d_norm = d_clean       
            
            ep_norm_data.append(d_norm)
            
            n_mice = len(d_norm)
            metadata["Data_Summary"][ep_name][group_label] = {
                "N_mice": n_mice,
                "Mean": float(np.mean(d_norm)) if n_mice > 0 else 0,
                "SEM": float(stats.sem(d_norm)) if n_mice > 0 else 0
            }            
            
            if n_mice > 0:
                local_max = np.max(d_norm)
                if local_max > global_max:
                    global_max = local_max
                    
        epoch_data_normalized.append(ep_norm_data)
        
    # 1.3 gives a 30% headroom buffer for double-stacked brackets.
    ax.set_ylim(0, global_max * 1.3) 

    # --- 3. PLOTTING LOOP ---
    for ep_idx in range(n_epochs):
        base_x = base_positions[ep_idx]
        ep_norm_data = epoch_data_normalized[ep_idx]
        
        # A. Plot Box and Scatter
        for grp_idx in range(n_groups):
            d_norm = ep_norm_data[grp_idx]
            if len(d_norm) == 0: continue
            
            x_pos = base_x + offsets[grp_idx]
            color = colors[grp_idx]
            
            # Transparent faces, solid edges
            face_color_rgba = mcolors.to_rgba(color, alpha=0.3)
            
            # Boxplot 
            ax.boxplot(d_norm, positions=[x_pos], widths=box_width, 
                       patch_artist=True, showfliers=False,
                       boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                       medianprops=dict(color=color, linewidth=1.0),
                       whiskerprops=dict(color=color, linewidth=0.5),
                       capprops=dict(color=color, linewidth=0.5))
            
            # Scatter (Tiny points with white borders)
            x_scatter = x_pos + np.random.uniform(-jitter_strength, jitter_strength, size=len(d_norm))
            ax.scatter(x_scatter, d_norm, s=2.5, color=color, 
                       alpha=1.0, edgecolors='white', linewidth=0.25, zorder=3)

        # B. Add Significance Brackets (MODIFIED FOR 2 GROUPS)
        if len(ep_norm_data) >= 2:
            d_grp1 = ep_norm_data[0]
            d_grp2 = ep_norm_data[1]
            
            local_ep_max = max([np.max(d) if len(d)>0 else 0 for d in ep_norm_data])
            
            x_grp1 = base_x + offsets[0]
            x_grp2 = base_x + offsets[1]
            # Bracket 1: Group 1 vs Group 2 (Single test)
            add_stat_annotation_two_sided(ax, d_grp1, d_grp2, x_grp1, x_grp2, local_ep_max, ttest=0, paired=0)      
            _, p_val = stats.mannwhitneyu(d_grp1, d_grp2, alternative='two-sided')
            metadata["Data_Summary"][epoch_names[ep_idx]]['p_val'] = p_val


    # --- FORMATTING ---
    ax.set_xticks(base_positions)
    ax.set_xticklabels(epoch_names, fontsize=6)
    
    # Consolidated to single line to prevent clipping, with labelpad buffer
    ax.set_ylabel('Cell Proportion (%)', labelpad=0.1) 
    
    # REPLACED hardcoded MultipleLocator with dynamic MaxNLocator
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=5))   
    
    # MODIFIED: Dynamically scale legend range to n_groups instead of hardcoded 3
    custom_lines = [Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], markersize=4, alpha=1.0) for i in range(n_groups)]
    ax.legend(custom_lines, labels, frameon=False, loc='upper left')    
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Changed from -0.5 to 0.5 to perfectly center the boxes on the canvas
    ax.set_xlim(0.5, n_epochs + 0.5)

    # --- STRICT EXPORTING ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    # Increased w_pad slightly (to 0.05 inches) to give the Y-label physical room to exist
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    #0.05    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## supp_2.8 Speed plot of EC3-C and EC3-I

In [ ]:
# Speed plot
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

base_du = 3 
post_du = 40 
bins_trial_start = int((20-base_du)*fs) 
bins_trial_end = int((20+20+20+3+ post_du)*fs) 
bins_speed_trial = int((20+160)*fs) # 20s base + 160 of the whole trial

epochs = {'Base':3 , 'CS': 20, 'Trace': 20, 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8}

speed_groups_dict = dict()
for i in range(group_size):    
    dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
    dpath_test = os.path.join(dpath_cal_group, test_algori_data)
    print(dpath_test) 
     # Read in speed info
    with open(os.path.join(dpath_test, 'Speed_bin_trials.pkl'), 'rb') as f:
        speed_bin_trial = pickle.load(f)   
    speed_mean_anmials = []
    for key in speed_bin_trial.keys():
        speed_tmp = speed_bin_trial[key].mean(axis=0) # Average all trials
        tmp = np.full(bins_speed_trial, np.nan) # To take care of nan 
        tmp[:speed_tmp.shape[-1]] = speed_tmp # Average from 6 trials
        speed_mean_anmials.append(tmp[bins_trial_start:bins_trial_end])
    speed_groups_dict[group_keys[i]] = np.stack(speed_mean_anmials)

width_mm = 40  
height_mm = 30 

plot_speed_groups(speed_groups_dict, 'Mean Speed (cm/s)', width_mm, height_mm, epochs, colors_beh_i, fs, dpath_plot, 'sup_02_8_Trial-Averaged speed in EC3_animal-wise')
print('All finished************')     

In [ ]:
def plot_speed_groups(speed_groups_dict, y_label, width_mm, height_mm, epochs, colors, fs, output_path, title):
    """
    Plots the average speed for different groups using a broken Y-axis 
    to accommodate the massive US escape response and the subtle CS/Trace freezing dynamics.
    
    speed_groups_dict: dict of {group_name: 2D np.array of shape (n_animals, n_bins)}
    """
    set_pub_style()    
    
    # 1. SETUP BROKEN AXES (Top for US burst, Bottom for CS/Trace dynamics)
    fig, (ax_top, ax_bottom) = plt.subplots(
        2, 1, 
        figsize=(width_mm / 25.4, height_mm / 25.4), 
        layout='constrained',
        sharex=True,
        gridspec_kw={'height_ratios': [1, 2], 'hspace': 0.05} 
    )
    
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # 2. TIME VECTOR GENERATION
    # Extract n_bins from the first available group array
    first_key = list(speed_groups_dict.keys())[0]
    n_bins = speed_groups_dict[first_key].shape[1]
    
    t_start = -epochs.get('Base', 3.0) 
    x_time = np.arange(n_bins) / fs + t_start

    # 3. PLOT DYNAMICS ON BOTH AXES
    for idx, (grp_name, grp_data) in enumerate(speed_groups_dict.items()):
        n_animals = grp_data.shape[0]
        if n_animals == 0: continue
            
        mean_curve = np.nanmean(grp_data, axis=0)
        sem_curve = np.nanstd(grp_data, axis=0) / np.sqrt(n_animals)
        
        c = colors[idx]
        
        for ax in (ax_top, ax_bottom):
            ax.plot(x_time, mean_curve, color=c, lw=0.75, label=grp_name, zorder=3)
            ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                            color=c, alpha=0.15, lw=0, zorder=2)
                            
        metadata["Data_Summary"][grp_name] = {"N_animals": int(n_animals)}

    # 4. EPOCH BACKGROUND SHADING
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),
        'Trace': ('gray', 0.05),
        'US': ('#e1703c', 0.15),
        'P_US1': ('gray', 0.05),
        'P_US2': ('gray', 0.15),
        'P_US3': ('gray', 0.20)
    }   
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue
            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            for ax in (ax_top, ax_bottom):
                ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)
                
        current_time += ep_dur

    # 5. Y-AXIS LIMITS & BROKEN EFFECT
    max_speed = 0
    for grp_data in speed_groups_dict.values():
        if grp_data.shape[0] > 0:
            local_max = np.nanmax(np.nanmean(grp_data, axis=0))
            max_speed = max(max_speed, local_max)
            
    top_limit = max(20.0, max_speed + 5.0) 
    
    ax_top.set_ylim(15, top_limit)
    ax_bottom.set_ylim(0, 5)

    # Hide the spines between the top and bottom axes
    ax_top.spines['bottom'].set_visible(False)
    ax_bottom.spines['top'].set_visible(False)
    ax_top.tick_params(labeltop=False, bottom=False)  
    ax_bottom.xaxis.tick_bottom()

    # Draw the diagonal hash marks (//)
    d = .015  
    kwargs = dict(transform=ax_top.transAxes, color='k', clip_on=False, lw=0.75)
    ax_top.plot((-d, +d), (-d, +d), **kwargs)        
    
    kwargs.update(transform=ax_bottom.transAxes)  
    ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)  

    # 6. FORMATTING
    ax_bottom.set_xlabel('Time from CS onset (s)', labelpad=1)
    # 1. Standardize the label without bolding
    ax_bottom.set_ylabel(y_label, labelpad=0.1)
    
    # 2. Push X further left to -0.18 or -0.20 to clear the tick numbers.
    #    Keep Y at 0.8 to keep it centered across the broken axis split.
    ax_bottom.yaxis.set_label_coords(-0.12, 0.8)
   
    for ax in (ax_top, ax_bottom):
        ax.set_xlim(t_start, x_time[-1])
        ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(0.5)
        ax.tick_params(axis='both', pad=1)
        
    ax_bottom.spines['bottom'].set_linewidth(0.5)
    
    # Ticks formatting
    ax_bottom.yaxis.set_major_locator(ticker.MultipleLocator(2))
    ax_top.yaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
    
    # Compact legend
    #ax_top.legend(frameon=False, ncol=len(speed_groups_dict), 
     #             loc='upper right', bbox_to_anchor=(1.05, 1.2))

    # 7. EXPORT
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(h_pad=0.0, hspace=0.0)
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

# Supplementary figure 5 for TFC-60, TFC-5, TFC-0 in EC3

## supp_3.1 Basic mean z score plot with different trace durations (TFC-60, TFC-5, TFC-0 in EC3)

In [ ]:
# TFC-60, TFC-5, TFC-0 plot
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

width_mm = 40  # 
height_mm = 30 #  

trace_groups = ['13.Post_TFC-60', '14.Post_TFC-5', '15.Post_TFC-0']
trace_durs = [60, 5, 0]
for idx_g, trace_key in enumerate(trace_groups):
    dpath_cal_all = f'{dpath_cal_supp}{trace_key}'
    trace_du = trace_durs[idx_g]

    base_du = 3 
    post_du = 40 
    bins_trial_start = int((20-base_du)*fs)
    bins_trial_end = int((20+20+trace_du+3+ post_du)*fs)
    cal_trial_animal = dict()
    for i in range(group_size):    
        dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
        dpath_test = os.path.join(dpath_cal_group, test_algori_data)
        print(dpath_test) 
        Sig_bin_trial_base = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials_base.nc"))    # The base is always 20s   
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))        
        bins_with_base = Sig_bin_trial_base.bins.size + Sig_bin_trial.bins.size
        Sig_bin_trial = xr.concat([Sig_bin_trial_base, Sig_bin_trial], dim='bins').assign_coords({'bins': range(bins_with_base)})#.drop_vars("session")  
        Sig_bin_trial = Sig_bin_trial.mean(dim='trials').sel(session='session0_condi')           
        #Extract data
        cal_trial_animal[group_keys[i]] = Sig_bin_trial['Sig_each_trial'].sel(bins=range(bins_trial_start, bins_trial_end)).values
        
    epochs = {'Base':3 , 'CS': 20, 'Trace': trace_du, 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8}
    plot_trial_population_activity_with_timeline(cal_trial_animal, width_mm, height_mm, epochs, group_keys, colors_beh_i, fs, dpath_plot, f'sup_03_1_{idx_g+1}_Trial-Averaged mean Z score-animal-wise-TFC-{trace_du}')
print('All finished************') 

In [ ]:
def plot_trial_population_activity_with_timeline(cal_trial, width_mm, height_mm, epochs, group_keys, colors, fs, output_path, title):
    """
    Plots the Trial-Averaged Population Activity for the entire TFC trial.
    Locked to strict millimeter layout (e.g., 60x30mm).
    Includes a floating timeline schematic mapped to shading colors/alphas, 
    with a continuous baseline spanning CS to US.
    """
    set_pub_style()
    fig, ax = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')  
    metadata = {"Figure_Title": title, "Data_Summary": {}}

    # --- 1. TIME VECTOR GENERATION ---
    # 0 is Tone Onset. Baseline represents negative time.
    t_start = -epochs.get('Base', 3.0) 
    
    # Extract number of bins from the first available dataset
    first_key = [k for k in group_keys if k in cal_trial][0]
    n_bins = cal_trial[first_key].shape[1]
    
    # Create time vector in SECONDS using the sampling frequency (fs)
    x_time = np.arange(n_bins) / fs + t_start

    # --- 2. DYNAMIC EPOCH SHADING & TIMELINE SCHEMATIC ---
    epoch_shading_map = {
        'CS': ('#4091cf', 0.1),       # Tone: Blue
        'Trace': ('gray', 0.05),      # Trace: Light Gray
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15),
        'P_US3': ('gray', 0.20)}      # Postw: dark Gray        
    
    current_time = t_start
    timeline_events = {}
    
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        
        # Track start and end coordinates for the schematic
        timeline_events[ep_name] = (current_time, current_time + ep_dur)
        
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # Floating Timeline Schematic (Skipped if Trace == 20)
    trace_dur = epochs.get('Trace', None)
    if trace_dur is not None and trace_dur != 20:
        # Increase space to PETH plot (1.15 pushes it nicely above the top spine)
        y_base = 1.05 
        if 'Trace' in timeline_events:
            trace_s, trace_e = timeline_events['Trace']
            trace_center = (trace_s + trace_e) / 2

            ax.plot([trace_s, trace_e], [y_base, y_base], transform=ax.get_xaxis_transform(),
                    color='black', lw=0.5, clip_on=False)
            # Place text just inside the gap above the baseline. 
            # If trace_dur == 0, it perfectly centers on the CS/US border intersection.
            ax.text(trace_center, y_base + 0.015, f"{int(trace_dur)} s", 
                    transform=ax.get_xaxis_transform(), ha='center', va='bottom', 
                    fontsize=7,  color='black', clip_on=False)
                    
    # --- 3. PLOT MEAN + SEM CURVES ---
    for i, grp in enumerate(group_keys):
        if grp not in cal_trial: continue
        
        data = cal_trial[grp]
        n_cells = data.shape[0]
        
        mean_curve = np.nanmean(data, axis=0)
        sem_curve = np.nanstd(data, axis=0) / np.sqrt(n_cells)
        color = colors[i]        
        
        # Micro-thin lines (0.75) to prevent ink crowding
        ax.plot(x_time, mean_curve, color=color, lw=0.75, label=f"{grp} (n={n_cells})", zorder=3)
        ax.fill_between(x_time, mean_curve - sem_curve, mean_curve + sem_curve, 
                        color=color, alpha=0.3, lw=0, zorder=2)

        metadata["Data_Summary"][grp] = {"N_mice": n_cells, "Peak_Z": float(np.nanmax(mean_curve))}

    # --- 4. FORMATTING ---
    ax.set_xlabel('Time from CS onset (s)', labelpad=1)
    ax.set_ylabel('Mean Z-Score', labelpad=0.1)
    
    # Strict limits based on exact epoch timings
    ax.set_xlim(t_start, x_time[-1])
    ax.xaxis.set_major_locator(ticker.MultipleLocator(20))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.4))
    
    # length=2 to ensure the physical tick marks stay tiny.
    ax.tick_params(axis='both')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(0.5)
    ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT ---
    os.makedirs(output_path, exist_ok=True)
    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## supp_3.2 Statistical plot of mean z score with different trace durations

In [ ]:
# TFC-60, TFC-5, TFC-0 plot
def extract_mean_data_among_group(df, group_keys, group_size):
    data_group =[]
    for i in range(group_size):
        data_group.append(df.loc[group_keys[i]].values.mean(axis=1))
    return data_group
    
group_name = ['05.EC3-C', '06.EC3-I']
group_keys = ['EC3-C', 'EC3-I']
group_size = len(group_name)

test_algori_data = 'post_01_8_basic_statistics_new'

width_mm = 40  # 
height_mm = 30 #  
test_keys = ['us','p_us1','p_us2' ] # p_us2 includes both p_us2 and p_us3
key_idx = {k: i+1 for i, k in enumerate(test_keys)}

trace_groups = ['13.Post_TFC-60', '14.Post_TFC-5', '15.Post_TFC-0']
trace_durs = [60, 5, 0]
for idx_g, trace_key in enumerate(trace_groups):
    dpath_cal_all = f'{dpath_cal_supp}{trace_key}'
    trace_du = trace_durs[idx_g]
    # initialize containers
    ds_cal_z = {k: [] for k in test_keys}
    for i in range(group_size):    
        dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
        dpath_test = os.path.join(dpath_cal_group, test_algori_data)
        print(dpath_test)    
        for k in test_keys:
            idx = key_idx[k]        
            # filenames
            cal_file = f"{idx}_Mean z-score trials in epoch-{k}.csv"
            ds_cal_z[k].append(pd.read_csv(os.path.join(dpath_test, cal_file)))
      
    # concat with group keys
    for k in test_keys:
        ds_cal_z[k] = pd.concat(ds_cal_z[k], keys=group_keys, names=["group", "row"])
    # 1. Average Z score plotting
    x_labels = ['US', 'pUS-1', 'pUS-2/3']
    ls_data_us = extract_mean_data_among_group(ds_cal_z['us'], group_keys, group_size)
    ls_data_p_us1 = extract_mean_data_among_group(ds_cal_z['p_us1'], group_keys, group_size)
    ls_data_p_us2 = extract_mean_data_among_group(ds_cal_z['p_us2'], group_keys, group_size)
    plot_multiparameter_z_score([ls_data_us, ls_data_p_us1, ls_data_p_us2], group_keys, x_labels, 'Mean Z-Score', width_mm, height_mm, colors_beh_i, 
                                dpath_plot, f'sup_03_2_{idx_g+1}_Trial-Averaged mean Z score-statitics with epochs in animal-wise_TFC-{trace_du}')

print('All finished************') 

In [ ]:
def plot_multiparameter_z_score(data_list, group_labels, x_labels, y_label, width_mm, height_mm, colors, output_path, title):
    """
    Plots a Micro-Panel Facet Grid for multi-epoch comparisons (e.g., US, p-US1, p-US2).
    - Clusters bars tightly together within each epoch panel.
    - Significance brackets dynamically hug the local maximum of each specific epoch.
    - Shares y-limits globally across the row.
    """
    set_pub_style()
    n_epochs = len(data_list)
    n_groups = len(group_labels)
    if n_groups not in [2, 3]:
        raise ValueError("This function is optimized for 2 or 3 animal groups per epoch.")
        
    # --- 1. SPACING LOGIC ---
    # Cluster the boxes closely around the center (0) of the x-axis
    if n_groups == 2:
        positions = [-0.25, 0.25]
        box_width = 0.35
        x_limits = (-0.8, 0.8) # Pads the edges so panels have visual distance between them
    else:
        positions = [-0.35, 0, 0.35]
        box_width = 0.25
        x_limits = (-0.8, 0.8)
    
    # Initialize unified figure layout canvas
    fig, axes = plt.subplots(1, n_epochs, figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    if n_epochs == 1: axes = [axes]
    
    metadata = {"Figure_Title": title, "Statistics": {}}

    # --- 2. GLOBAL UNIFIED Y-AXIS SCALING ---
    all_vals_flat = [val for epoch in data_list for grp in epoch for val in grp if not np.isnan(val)]
    if len(all_vals_flat) == 0:
        raise ValueError("No valid scalar data found across parameters.")
        
    global_max = max(all_vals_flat)
    global_min = min(all_vals_flat)
    y_range = global_max - global_min
    
    # Compute shared bounds for the scale itself
    shared_ymin = global_min - (y_range * 0.05)
    shared_ymax = global_max + (y_range * 0.35)

    # --- 3. ITERATE SUBPLOTS THROUGH THE ROW ---
    for idx, ax in enumerate(axes):
        epoch_data = data_list[idx]
        current_x_label = x_labels[idx]
        metadata["Statistics"][current_x_label] = {}
        
        # Apply strict uniform y-constraints to the axis
        ax.set_ylim(shared_ymin, shared_ymax)
        ax.set_xlim(x_limits)
        
        # Calculate LOCAL maximum for this specific epoch to keep brackets close to the data
        local_vals = [val for grp in epoch_data for val in grp if not np.isnan(val)]
        local_max = max(local_vals) if len(local_vals) > 0 else 0
        
        # The gap above the box uses the GLOBAL y_range so the physical gap size looks identical across panels
        current_roof = local_max + (y_range * 0.06)
        
        # A. Plot Background Boxplots and Strip Points
        for grp_idx in range(n_groups):
            grp_data = epoch_data[grp_idx]
            clean_data = grp_data[~np.isnan(grp_data)]
            n_animals = len(clean_data)
            
            metadata["Statistics"][current_x_label][group_labels[grp_idx]] = {"N_animals": n_animals}
            if n_animals == 0: continue
            
            x_pos = positions[grp_idx]
            color = colors[grp_idx]
            face_color_rgba = mcolors.to_rgba(color, alpha=0.4)
            
            ax.boxplot(clean_data, positions=[x_pos], widths=box_width, 
                       patch_artist=True, showfliers=False, zorder=1,
                       boxprops=dict(facecolor=face_color_rgba, edgecolor=color, linewidth=0.5),
                       medianprops=dict(color=color, linewidth=1.0),
                       whiskerprops=dict(color=color, linewidth=0.5),
                       capprops=dict(color=color, linewidth=0.5))
            
            jittered_x = np.random.uniform(x_pos - box_width/4, x_pos + box_width/4, size=n_animals)
            ax.scatter(jittered_x, clean_data, s=4, color=color, alpha=1.0, 
                       edgecolors='white', linewidth=0.25, zorder=2)

        # B. Independent Significance Annotations (using the new local positions)
        if len(epoch_data[0]) > 2 and len(epoch_data[1]) > 2:
            current_roof = add_stat_annotation_two_sided(
                ax, epoch_data[0], epoch_data[1], 
                positions[0], positions[1], current_roof, ttest=0, paired=0)            
            _, p_mwu = stats.mannwhitneyu(epoch_data[0], epoch_data[1], alternative='two-sided')
            metadata["Statistics"][current_x_label][f"{group_labels[0]}_vs_{group_labels[1]}"] = float(p_mwu)
            
        if n_groups == 3 and len(epoch_data[0]) > 2 and len(epoch_data[2]) > 2:
            current_roof += (y_range * 0.08)
            add_stat_annotation_two_sided(
                ax, epoch_data[0], epoch_data[2], 
                positions[0], positions[2], current_roof, ttest=0, paired=0)
            
            _, p_mwu = stats.mannwhitneyu(epoch_data[0], epoch_data[2], alternative='two-sided')
            metadata["Statistics"][current_x_label][f"{group_labels[0]}_vs_{group_labels[2]}"] = float(p_mwu)

        # --- 4. MICRO-PANEL FACET GRAPH AESTHETICS ---
        ax.set_xticks([])
        ax.set_xticklabels([])
        ax.set_xlabel(current_x_label, labelpad=2)

        #fig.supxlabel('Conditioning epochs')
        # Shared Left Margin Logic
        if idx == 0:
            ax.set_ylabel(y_label,  labelpad=1)
            ax.yaxis.set_major_locator(ticker.MultipleLocator(0.4))
            ax.tick_params(axis='y', length=2, pad=1)
            ax.spines['left'].set_linewidth(0.5)
        else:
            ax.set_ylabel('')
            ax.set_yticks([])
            ax.set_yticklabels([])
            ax.tick_params(axis='y', length=0)
            ax.spines['left'].set_visible(False)
            
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_linewidth(0.5)

    # --- 5. EXPORT MANUSCRIPT GRAPHIC ---
    base_path = os.path.join(output_path, title.replace(' ', '_'))
    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

## supp_3.3 Statistical plot of participation Ratio (PR) with different trace durations

In [ ]:
# TFC-60, TFC-5, TFC-0 plot    
group_name = ['05.EC3-C', '06.EC3-I', '01.EC5b']
group_keys = ['EC3-C', 'EC3-I', 'EC5b']
group_size = len(group_name)
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

width_mm = 40  # 
height_mm = 30 #  

trace_groups = ['13.Post_TFC-60', '14.Post_TFC-5', '15.Post_TFC-0']
trace_durs = [60, 5, 0]
for idx_g, trace_key in enumerate(trace_groups):
    dpath_cal_all = f'{dpath_cal_supp}{trace_key}'
    trace_du = trace_durs[idx_g]

    base_du = 3 
    post_du = 20 
    bins_trial_start = int((20+20+trace_du-base_du)*fs) 
    bins_trial_end = int((20+20+trace_du+3+ post_du)*fs)
    
    cal_trial_cells = []
    for i in range(group_size):    
        dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
        dpath_test = os.path.join(dpath_cal_group, test_algori_data)
        print(dpath_test) 
        Sig_bin_trial_base = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials_base_cells_pool.nc")) # The base is always 20s   
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials_cells_pool.nc"))        
        bins_with_base = Sig_bin_trial_base.bins.size + Sig_bin_trial.bins.size
        Sig_bin_trial = xr.concat([Sig_bin_trial_base, Sig_bin_trial], dim='bins').assign_coords({'bins': range(bins_with_base)})#.drop_vars("session")     
        
        Sig_bin_trial = Sig_bin_trial['Sig_each_trial'].sel(bins=range(bins_trial_start, bins_trial_end))   
        cal_trial_cells.append(Sig_bin_trial.values.transpose(1, 0, 2)) ## shape to (n_cells,n_trials, n_bins)
    
    #Plot the PR curve
    ds_c = cal_trial_cells[0]
    ds_i = cal_trial_cells[1]
    ds_source = cal_trial_cells[2]
    t, pr_c, pr_i, rate = analyze_dimensionality_concatenated(ds_c, ds_i, ds_source, fs=5.0, window_sec=3.0, step_sec=0.2) 
    
    epochs = {'Base':3 , 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8}
    plot_pr_constriction(t, pr_c, pr_i, group_keys, epochs, width_mm, height_mm, colors_beh_i, dpath_plot, f'sup_03_3_{idx_g+1}_population level-Participation ratio_TFC-{trace_du}')
     
print('All finished************') 

In [ ]:
def plot_pr_constriction(time_vec, pr_c, pr_i, group_labels, epochs, width_mm, height_mm, colors, output_path, title):
    """
    Plots the Participation Ratio (Effective Dimensionality) over time.
    Utilizes the pre-calculated time_vec and shifts it so the Base epoch is negative,
    aligning exactly 0 with US onset.
    Locked to strict millimeter layout.
    """
    set_pub_style()   
    # 1. Safely flatten inputs to 1D arrays
    time_vec_flat = np.array(time_vec).flatten()
    pr_c_flat = np.array(pr_c).flatten()
    pr_i_flat = np.array(pr_i).flatten()
    
    if not (time_vec_flat.size == pr_c_flat.size == pr_i_flat.size):
        raise ValueError("time_vec, pr_c, and pr_i must have identical sizes.")

    metadata = {
        "Figure_Title": title,
        "Statistics": {
            "N_bins": int(pr_c_flat.size)
        }
    }
    # --- 2. CANVAS SETUP ---
    fig, ax1 = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')
    
    # --- 3. TIME VECTOR SHIFTING ---
    # Shift the pre-calculated time_vec so the Base epoch becomes negative,
    # placing the US onset perfectly at 0.
    t_base = epochs.get('Base', 3.0)
    shifted_time_vec = time_vec_flat - t_base

    # --- 4. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
        'US': ('#e1703c', 0.15),      # Shock: Orange
        'P_US1': ('gray', 0.05),      # Post1: Light Gray
        'P_US2': ('gray', 0.15),       # Post2: Dark Gray
        'P_US3': ('gray', 0.20)
    }
    
    # Start shading counter at the negative base time (e.g., -3.0)
    current_time = -t_base
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax1.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur

    # --- 5. PLOT DIMENSIONALITY (PR) ---
    ax1.plot(shifted_time_vec, pr_c_flat, color=colors[0], lw=1.0, 
             label=group_labels[0], zorder=3)
    ax1.plot(shifted_time_vec, pr_i_flat, color=colors[1], lw=1.0, 
             label=group_labels[1], zorder=3)

    # --- 6. FORMATTING, TICKS & SPINES ---
    ax1.set_xlabel('Time from US onset (s)', labelpad=1)
    ax1.set_ylabel('Dimensionality (PR)', labelpad=1)
    
    # Strict limits based on the actual shifted time vector
    ax1.set_xlim(shifted_time_vec[0], shifted_time_vec[-1])
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
    
    # Limit max ticks to prevent Y-axis crowding in a 30mm height
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(1))
    
    ax1.tick_params(axis='both', length=2, pad=1, colors='black')
    
    ax1.spines['left'].set_linewidth(0.5)
    ax1.spines['bottom'].set_linewidth(0.5)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)


    base_path = os.path.join(output_path, title.replace(' ', '_'))   
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)  
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()
    
    save_metadata_json(metadata, output_path, title)

def get_participation_ratio(data_2d):
    """
    Calculates the Participation Ratio (PR) of a 2D neural population matrix.
    Automatically uses the Gram matrix trick for speed if N_cells > N_timepoints.
    """
    n_cells, n_time = data_2d.shape   
    # 1. Mean-center the data for each cell across the concatenated time vector
    X = data_2d - np.mean(data_2d, axis=1, keepdims=True)  
    # 2. Calculate Covariance Eigenvalues
    if n_cells > n_time:
        # Gram Matrix trick: Eigenvalues of (X.T @ X) are the same as (X @ X.T)
        # Scale doesn't strictly matter for PR since it's a ratio, but keep it mathematically correct.
        cov = np.dot(X.T, X) / (n_time - 1)
    else:
        # Standard Covariance
        cov = np.dot(X, X.T) / (n_time - 1)       
    # Get Eigenvalues (eigh is optimized for symmetric matrices)
    u = np.linalg.eigvalsh(cov)   
    # Filter out numerical noise (tiny or negative float artifacts)
    u = u[u > 1e-9]   
    # 3. PR Formula
    if len(u) == 0:
        return 0.0       
    pr = (np.sum(u) ** 2) / np.sum(u ** 2)
    return pr

def analyze_dimensionality_concatenated(ec3_c, ec3_i, ec5b, fs=5.0, window_sec=3.0, step_sec=0.2):
    """
    Performs sliding-window dimensionality analysis using trial concatenation.
    
    Parameters:
    -----------
    ec3_3d : np.array (n_cells_ec3, n_trials, n_total_bins)
    ec5b_3d : np.array (n_cells_ec5b, n_trials, n_total_bins)
    fs : float
        Sampling rate in Hz (0.2s bins = 5.0 Hz).
    window_sec : float
        Width of the sliding window.
    step_sec : float
        Step size to advance the window.
    """
    n_cells_ec3_c, n_trials, n_total_bins = ec3_c.shape
    n_cells_ec3_i = ec3_i.shape[0]
    window_bins = int(window_sec * fs)
    step_bins = int(step_sec * fs)
    
    time_points = []
    pr_ec3_trace_c = []
    pr_ec3_trace_i = []
    rate_ec5b_trace = []
    
    # Sliding Window Loop
    for t_start in range(0, n_total_bins - window_bins + 1, step_bins):
        t_end = t_start + window_bins
        
        # 1. Slice the 3D window
        ec3_c_window = ec3_c[:, :, t_start:t_end]
        ec3_i_window = ec3_i[:, :, t_start:t_end]
        ec5b_window = ec5b[:, :, t_start:t_end]
        
        # 2. Concatenate Trials (Flatten dimensions 1 and 2)
        # Shape goes from (2000, 6, 15) -> (2000, 90)
        ec3_c_concat = ec3_c_window.reshape(n_cells_ec3_c, -1)
        ec3_i_concat = ec3_i_window.reshape(n_cells_ec3_i, -1)
        # 3. Calculate Dimensionality
        pr = get_participation_ratio(ec3_c_concat)
        pr_ec3_trace_c.append(pr)
        pr = get_participation_ratio(ec3_i_concat)
        pr_ec3_trace_i.append(pr)
        # 4. Calculate EC5b Population Rate
        # Average across all cells, trials, and bins in this window
        rate = np.mean(ec5b_window)
        rate_ec5b_trace.append(rate)
        
        # Record Time (Center of the window)
        time_points.append((t_start + t_end) / 2.0 / fs)
        
    return np.array(time_points), np.array(pr_ec3_trace_c), np.array(pr_ec3_trace_i), np.array(rate_ec5b_trace)

## supp_3.4 "Braking effect" plot with different trace durations (60s, 5s, 0s)

In [ ]:
# TFC-60, TFC-5, TFC-0 plot    
group_name = ['05.EC3-C', '06.EC3-I', '01.EC5b']
group_keys = ['EC3-C', 'EC3-I', 'EC5b']
group_size = len(group_name)
test_algori_data = 'post_01_3_overview_QC_z_score_condi_bin_0.2s_new_z'

width_mm = 40  # 
height_mm = 30 #  

trace_groups = ['13.Post_TFC-60', '14.Post_TFC-5', '15.Post_TFC-0']
trace_durs = [60, 5, 0]
for idx_g, trace_key in enumerate(trace_groups):
    dpath_cal_all = f'{dpath_cal_supp}{trace_key}'
    trace_du = trace_durs[idx_g]

    base_du = 3 
    post_du = 20 
    bins_trial_start = int((20+20+trace_du-base_du)*fs) 
    bins_trial_end = int((20+20+trace_du+3+ post_du)*fs) 
    
    cal_trial_animal = []
    for i in range(group_size):    
        dpath_cal_group = os.path.join(dpath_cal_all, group_name[i])
        dpath_test = os.path.join(dpath_cal_group, test_algori_data)
        print(dpath_test) 
        Sig_bin_trial_base = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials_base.nc")) # The base is always 20s   
        Sig_bin_trial = xr.open_dataset(os.path.join(dpath_test, "Cal_bin_trials.nc"))        
        bins_with_base = Sig_bin_trial_base.bins.size + Sig_bin_trial.bins.size
        Sig_bin_trial = xr.concat([Sig_bin_trial_base, Sig_bin_trial], dim='bins').assign_coords({'bins': range(bins_with_base)})#.drop_vars("session")  
        Sig_bin_trial = Sig_bin_trial.mean(dim='trials').sel(session='session0_condi')           
        cal_trial_animal.append(Sig_bin_trial['Sig_each_trial'].mean(dim='animal').sel(bins=range(bins_trial_start, bins_trial_end)).values)
   
    colors = ['black','#A6761D'] # 1st is for the delta of I and C, 2nd color is for EC5b
    epochs = {'Base':3 , 'US': 3, 'P_US1': 3, 'P_US2': 6, 'P_US3': 8}
    plot_overshoot_brake(cal_trial_animal, group_keys, epochs, fs, width_mm, height_mm, colors, dpath_plot, f'sup_03_4_{idx_g+1}_EC5b to EC3 braking effect_TFC-{trace_du}')
    
print('All finished************') 

In [ ]:
def plot_overshoot_brake(ls_data, group_labels, epochs, fs, width_mm, height_mm, colors, output_path, title):
    """
    Analyzes and plots the "Braking Effect" using the Subtraction Method.
    Calculates Delta Activity (EC3-I - EC3-C) representing "Lost Inhibition" 
    and cross-correlates it directly with the recorded EC5b source activity.
    Locked to strict millimeter layout
    """
    set_pub_style()
    data_c, data_i, source = ls_data[0], ls_data[1], ls_data[2]
    # 1. Safely flatten inputs to 1D arrays
    target_c = np.array(data_c).flatten()
    target_i = np.array(data_i).flatten()
    source_activity = np.array(source).flatten()
    
    if not (target_c.size == target_i.size == source_activity.size):
        raise ValueError("Time bins for data_c, data_i, and source must be identical sizes.")

    # --- 2. THE SUBTRACTION METHOD CALCULATION ---
    # Delta represents the runaway activity unmasked when the brake is lost
    delta_activity = target_i - target_c
    
    # Compute Pearson cross-correlation coefficient to statistically evaluate coupling
    r_coeff, p_value = stats.pearsonr(delta_activity, source_activity)

    metadata = {
        "Figure_Title": title,
        "Statistics": {
            "Pearson_r": float(r_coeff),
            "Pearson_p": float(p_value),
            "N_bins": int(delta_activity.size)}}
    # --- 3. CANVAS SETUP ---
    fig, ax1 = plt.subplots(figsize=(width_mm / 25.4, height_mm / 25.4), layout='constrained')    
    # Establish time vector. Shifted so 0 aligns with Shock Onset
    time_vec = np.arange(delta_activity.size) / fs - 3.0

    # --- 4. DUAL AXIS PLOTTING ---
    # Use twinx to overlay both variables without squashing their distinct amplitudes
    ax2 = ax1.twinx()    
    # Plot Delta Activity on the Left Axis (ax1)
    ax1.plot(time_vec, delta_activity, color=colors[0], lw=0.75, 
             label=f"$\Delta$ ({group_labels[1]} - {group_labels[0]})", zorder=3)    
    # Plot EC5b Source Activity on the Right Axis (ax2)
    ax2.plot(time_vec, source_activity, color=colors[1], lw=0.75, 
             label=f"{group_labels[2]} Activity", zorder=3)

    # --- 5. OVERLAYS & REFERENCE LINES ---
    t_start = -epochs.get('Base', 3.0)     
    # --- 2. DYNAMIC EPOCH SHADING ---
    epoch_shading_map = {
            'US': ('#e1703c', 0.15),      # Shock: Orange
            'P_US1': ('gray', 0.05),      # Post1: Light Gray
            'P_US2': ('gray', 0.15),
            'P_US3': ('gray', 0.20)}       # Postw: dark Gray       
    current_time = t_start
    for ep_name, ep_dur in epochs.items():
        if ep_name == 'Base':
            current_time += ep_dur
            continue # Base is unshaded white space            
        if ep_name in epoch_shading_map:
            c, a = epoch_shading_map[ep_name]
            ax1.axvspan(current_time, current_time + ep_dur, color=c, alpha=a, lw=0, zorder=0)            
        current_time += ep_dur
        
    # Zero baseline reference for the Delta Activity
    ax1.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.8, zorder=1)

    # Print the r-value overlay securely inside the plot canvas window
    p_string = f"p < 0.001" if p_value < 0.001 else f"p = {p_value:.3f}"
    stat_text = f"Pearson $r$ = {r_coeff:.2f}\n{p_string}"
    ax1.text(0.35, 0.90, stat_text, transform=ax1.transAxes, 
             fontsize=6, va='top', ha='left', zorder=4)

    # --- 6. FORMATTING, TICKS & SPINES ---
    ax1.set_xlabel('Time from US onset (s)', labelpad=1)
    ax1.set_ylabel('$\Delta$ Activity (Z-Score)', color=colors[0], labelpad=0.1)
    ax2.set_ylabel(f'{group_labels[2]} (Z-Score)', color=colors[1], labelpad=2)
    
    ax1.set_xlim(time_vec[0], time_vec[-1])
    ax1.xaxis.set_major_locator(ticker.MultipleLocator(5))
    
    # Limit max ticks to prevent Y-axis crowding
    ax1.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
    ax2.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    
    # Match spine and tick colors to their respective plot line identities
    ax1.tick_params(axis='y', length=2, pad=1, colors=colors[0])
    ax1.tick_params(axis='x', length=2, pad=1, colors='black') # X-axis remains black
    ax2.tick_params(axis='y', length=2, pad=1, colors=colors[1])
    
    ax1.spines['left'].set_color(colors[0])
    ax1.spines['left'].set_linewidth(0.5)
    ax2.spines['right'].set_color(colors[1])
    ax2.spines['right'].set_linewidth(0.5)
    ax1.spines['bottom'].set_linewidth(0.5)
    
    # Clean up upper and inner boundaries
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax2.spines['bottom'].set_visible(False) # Prevent bottom spine double-drawing

    base_path = os.path.join(output_path, title.replace(' ', '_'))    
    fig.set_constrained_layout_pads(w_pad=0.01, h_pad=0.01, hspace=0, wspace=0)    
    # Saved with default opaque backdrops as requested
    plt.savefig(f"{base_path}.pdf")
    plt.savefig(f"{base_path}.png", dpi=300)
    plt.close()    
    save_metadata_json(metadata, output_path, title)